# CS-781 Phase 7 — Prior Case Retrieval (notebook K)

Given a query case and its split's candidate pool, return the **7** most relevant prior cases (Micro-F1@7). Same
structure as Phases 2-6: your Phase-1 models + **your** N6 code (full / partial / BitFit / LoRA / LoRMA / rsLoRA / IA³ /
prompt tuning / distillation — copied unchanged from Phase 6, plus two functions you write: **`info_nce_loss`** and
**`kd_scores_loss`**), a training script, an **embedding** script, and a **lab** that turns everything into Codabench
zips plus a table and chart of tentative scores.

**How a case becomes a vector.** A judgment is far longer than the 1024 (encdec 768) tokens your models read. Every
document is cut into windows of `win_tokens` (512); each window goes through your Phase-1 model (no output layer: the
encoder states of encdec, the bidirectional prefix of prefix, causal states of decoder / swa / moe), the hidden states
are averaged over the tokens and normalised. Training: query window ↔ window of a relevant case, with BM25 hard negatives
and in-batch negatives (`info_nce_loss`); the distillation student also matches a teacher's similarity distribution
(`kd_scores_loss`). Inference: up to `embed_windows` windows per document, combined in several ways (first window, mean
embedding, MaxSim over windows, best window pair).

**No test labels, no extra data.** The dev set (labels allowed for model selection) picks each run's checkpoint — a small
re-ranking task on the first `val_queries` dev queries — and scores every variant. `dev_f1_rest` (dev queries that did
not pick a checkpoint) is the cleaner number; the lab still picks its best variants on dev, so expect a little optimism.

**Many zips:** every embedded run × window aggregation × hubness correction (CSLS), the best of each run fused with BM25
(a lexical score over your Phase-1 tokens — cases share statute names and phrases, so a lexical signal is strong), and
ensembles of the k best runs → e.g. 8 runs × 5 × 3 + fusion + ensembles ≈ 200 prediction sets, of which the 40 best (+
the best of each run) are zipped. `submissions/index.csv` + `submissions/score_chart.png` list them; **you choose what to
upload**. A BM25-only row is printed as a reference and never zipped (it is not a Phase-1 model). The dense-only rows
are what your fine-tuning actually learned: compare them with the fused rows in your report. Settings marked YOUR CHOICE
are yours (rule §17.5); declare that the pipeline / embedding / lab / chart code was written with an assistant.

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** (Hugging Face download) · Add-ons → Secrets →
**`Capstone_Phase_1` attached**.

**Inputs (Add Input)** — the same five as Phase 6:
| Input | Provides |
|---|---|
| `notebook857159535b` | tokenizer (`bpe_tok.py`, `bpe.json`) |
| `notebook27acca3fc3` (notebook D, phase1-continue) | **decoder** checkpoint `continued/best_continued.pt` |
| `Phase1_Training1` | encdec, swa checkpoints |
| dataset `prefix` | prefix checkpoint |
| `Phase1_training2` (phase1-train-extra) | moe (the distillation teacher) |
| *optional:* the output of an earlier run of **this** notebook | its runs / embeddings join the lab; its `p7cache` is reused |

**How to use:** write the two functions in N6 (section 9), then Save Version → Save & Run All (~5 h). N7b stops within a
minute if an input is missing or a function is not written. Then Output → `submissions/`: `index.csv`, `score_chart.png`
and the zips; download them into your `Phase 7` folder. Phase 7 closes **7 November 23:59** and Codabench needs 15–30
minutes per submission: start the run now, and keep a day for uploading.

## N1. Login, download the Phase 7 data

In [ ]:
import os, shutil, json
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
os.makedirs("/kaggle/working/p7data", exist_ok=True)
for path in ["Train/train.json", "Train/train_candidates.json", "Dev/dev.json", "Dev/dev_candidates.json",
             "Test/test.json", "Test/test_candidates.json"]:
    src = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_7_PCR/{path}", repo_type="dataset", token=HF_TOKEN)
    shutil.copy(src, f"/kaggle/working/p7data/{os.path.basename(path)}")
print({f: f"{os.path.getsize('/kaggle/working/p7data/' + f) / 1e6:.1f} MB" for f in sorted(os.listdir("/kaggle/working/p7data"))})

## N2. GPU check

In [ ]:
import torch
print("GPU count:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## N3. Configuration — experiments, retrieval options, lab settings

In [ ]:
import json
# =====================================================================================================
#  N3 — CONFIGURATION. Values marked YOUR CHOICE are yours (course rule §17.5); the comments explain each one.
# =====================================================================================================
#
#  PHASE 7 RULES THIS CONFIG MUST SATISFY (§13)
#  * every experiment starts from a Phase-1 model; no other pretrained weights, no external APIs
#  * ≥ 4 substantially different experiments: ≥ 3 different techniques FROM THE §13.2.1 LIST — Full fine-tuning,
#    Partial fine-tuning, QLoRA, LORMA, Prefix Tuning, Prompt Tuning, IA³, Knowledge Distillation — LORMA COMPULSORY —
#    and ≥ 1 knowledge distillation. NOTE: plain LoRA and rsLoRA are NOT on this list. Pooling, similarity and ranking
#    choices (this notebook's lab) do not count as techniques.
#    The starting plan covers: Full (moe_full = the teacher), LoRMA (encdec_lorma, prefix_lorma), IA³ (prefix_ia3),
#    Prompt Tuning (swa_prompt), Partial (decoder_partial), Knowledge Distillation (kd_moe_to_prefix4,
#    kd_moe_to_decoder4).
#  * Codabench rule: submissions that differ only in hyper-parameters / decoding-style settings do not count as
#    different experiments — upload zips from DIFFERENT runs (different technique / architecture) for the minimum.

CONFIG = {
    "seed": 42,
    "checkpoints": {
        "decoder":   "continued/best_continued.pt",
        "prefix":    "prefix/best.pt",
        "encdec":    "encdec/best.pt",
        "swa":       "swa/best.pt",
        "moe":       "moe/best.pt",
    },
    # checkpoint selection (dev labels are allowed for model selection): a re-ranking task on the first `val_queries`
    # dev queries, candidates = their relevant cases + the `minipool_k` best BM25 others, first window of each
    "val_queries": 60,           # YOUR CHOICE
    "minipool_k": 30,            # YOUR CHOICE
    "hn_pool": 100,              # BM25 hard-negative pool per training query (best first)
    "bm25": {"k1": 1.2, "b": 0.75},   # YOUR CHOICE. BM25 over Phase-1 tokens: hard negatives + the lexical score in the lab

    # ---------------- defaults for every experiment (an experiment below can override any of them) ----------------
    "defaults": {
        # --- retrieval-specific ---
        "win_tokens": 512,       # YOUR CHOICE. tokens per window (capped by the model: max_len - prompt tokens)
        "token_pool": "mean",    # YOUR CHOICE. how a window's token states become one vector: "mean" / "max" / "last"
        "temperature": 0.05,     # YOUR CHOICE. InfoNCE temperature (cosine scores are divided by it)
        "batch_queries": 6,      # YOUR CHOICE. queries per step; each step encodes B * (2 + hard_negs) windows
        "hard_negs": 2,          # YOUR CHOICE. BM25 / random negatives per query per step (+ in-batch negatives)
        "hn_range": [3, 60],     # YOUR CHOICE. negatives are drawn from BM25 ranks lo..hi of the pool (skipping the very
                                 #   top ranks lowers the chance of an unlabeled relevant case being used as a negative)
        "hard_frac": 0.75,       # YOUR CHOICE. share of negatives from BM25 ranks (the rest: random pool cases)
        # --- compute ---
        "time_budget_min": 40,   # YOUR CHOICE. training minutes per run (or `epochs`, whichever ends first)
        "eval_minutes": 12,      # dev re-ranking check every this many minutes (and at each epoch end)
        "epochs": 4,
        # --- the rest as in Phases 2-6 ---
        "lr": 5e-5,              # YOUR CHOICE
        "warmup_frac": 0.1,
        "weight_decay": 0.01,
        "grad_clip": 1.0,
        "method_cfg": {},
        "moe_aux_coef": 0.0,
        "teacher": None,
        "kd": {"temperature": 0.05, "alpha": 0.5},   # YOUR CHOICE. loss = alpha * InfoNCE + (1 - alpha) * KD
        "student_layers": None,
    },

    # ---------------- the experiments — YOUR CHOICE ----------------
    "experiments": {
        "moe_full":           {"arch": "moe", "method": "full", "batch_queries": 4},   # the distillation teacher
        "kd_moe_to_prefix4":  {"arch": "prefix", "method": "full", "teacher": "moe_full", "student_layers": 4},
        "kd_moe_to_decoder4": {"arch": "decoder", "method": "full", "teacher": "moe_full", "student_layers": 4},
        "encdec_lorma":       {"arch": "encdec", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "prefix_lorma":       {"arch": "prefix", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "prefix_ia3":         {"arch": "prefix", "method": "ia3", "lr": 5e-3},
        "swa_prompt":         {"arch": "swa", "method": "prompt", "method_cfg": {"n_tokens": 20}, "lr": 3e-3},
        "decoder_partial":    {"arch": "decoder", "method": "partial", "method_cfg": {"n_last": 2}, "lr": 1e-4},
        # --- optional extras (not in SESSION_EXPERIMENTS below): ---
        "prefix_full":        {"arch": "prefix", "method": "full"},
        "decoder_lorma":      {"arch": "decoder", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "swa_lorma":          {"arch": "swa", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "encdec_full":        {"arch": "encdec", "method": "full"},
        "prefix_frozen":      {"arch": "prefix", "method": "none"},    # zero-shot: the Phase-1 model, no fine-tuning (analysis)
        "decoder_frozen":     {"arch": "decoder", "method": "none"},
    },
}

# ---------------- what THIS session does on the GPUs ----------------
SESSION_EXPERIMENTS = ["moe_full", "prefix_lorma", "encdec_lorma", "kd_moe_to_prefix4", "swa_prompt", "prefix_ia3",
                       "decoder_partial", "kd_moe_to_decoder4"]
SMOKE_EXPERIMENTS = list(SESSION_EXPERIMENTS)
EMBED_RUNS = list(SESSION_EXPERIMENTS)   # runs that get embedded (a run from an earlier session's output can be added)

# ---------------- lab — YOUR CHOICE ----------------
LAB = {
    "embed_windows": 12,         # windows kept per document at inference (spread over the whole case)
    "aggs": ["first", "mean", "maxsim_q", "maxsim_sym", "maxall"],   # how window vectors become a query-candidate score
    "csls": [0, 0.5, 1.0],       # hubness correction strengths: score - beta * (candidate's mean similarity to its k nearest queries)
    "csls_k": 10,
    "fusion_w": [0.25, 0.5, 1, 2],   # BM25 weights in dense + w * BM25 (both z-scored per query); 0 = dense only is always scored
    "ensembles": True,           # z-score average of the k best runs, k = 2..n
}
TOP_N_ZIPS = 40              # zips for the 40 best variants (+ the best variant of every run)

CONFIG["lab"] = LAB
json.dump(CONFIG, open("/kaggle/working/run_config_pcr.json", "w"), indent=1)
for name, e in CONFIG["experiments"].items():
    assert e["arch"] in CONFIG["checkpoints"], f"{name}: unknown arch {e['arch']}"
    t = e.get("teacher")
    assert not t or t in CONFIG["experiments"], f"{name}: teacher {t} is not an experiment"
for name in SESSION_EXPERIMENTS:
    assert name in CONFIG["experiments"], f"SESSION_EXPERIMENTS: {name} is not defined"
assert CONFIG["defaults"]["hn_range"][1] <= CONFIG["hn_pool"], "hn_range must lie inside hn_pool"
budget = [CONFIG["experiments"][e].get("time_budget_min", CONFIG["defaults"]["time_budget_min"]) for e in SESSION_EXPERIMENTS]
print(f"{len(CONFIG['experiments'])} experiments defined; this session trains {len(SESSION_EXPERIMENTS)}: {SESSION_EXPERIMENTS}")
n_agg = len(LAB["aggs"]) * len(LAB["csls"])
print(f"lab: {len(EMBED_RUNS)} runs x {n_agg} dense variants (+ fusion with {len(LAB['fusion_w'])} BM25 weights, + ensembles)")
print(f"rough GPU time: ~{(sum(b + 4 for b in budget) + 15 * len(EMBED_RUNS)) / 2 / 60:.1f} h on T4 x2 "
      "(training + embedding; embedding time depends on the pool sizes; + ~15 min lab)")

## N4. Pipeline code: data, tokenisation cache, BM25, windows, window encoder, metric, files

In [ ]:
%%writefile pcr_common.py
"""Phase 7 (prior case retrieval) pipeline: data (query cases + one candidate pool per split), one-time Phase-1
tokenisation (cached on disk), BM25 over Phase-1 tokens (hard-negative mining + a lexical reference), fixed-size windows
of long documents, your Phase-1 model as a window encoder (hidden states pooled over tokens, L2-normalised), the
retrieval metric (Micro-F1@7, as in evaluate_phase7.py), run finder, files."""
import glob, json, os, sys
from multiprocessing import Pool
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.nn.utils import parametrize

INPUT_ROOT = os.environ.get("P7_INPUT", "/kaggle/input")
WORK = os.environ.get("P7_WORK", "/kaggle/working")
DATA = f"{WORK}/p7data"
CACHE = f"{WORK}/p7cache"
SPLITS = ("train", "dev", "test")
K = 7                    # evaluation depth: exactly 7 candidates per query
LEN_BUCKET = 32          # a window shorter than the window size is cut to a multiple of this (fewer distinct lengths)
DOC_CAP = 24000          # a document keeps at most this many tokens (memory bound only; very rare)

def find_input(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{pattern}' not found under {INPUT_ROOT} — is the right notebook/dataset added as input?")
    return hits[0]

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

_BPE = None
def bpe_tok_bpe():
    global _BPE
    if _BPE is None:
        _BPE = bpe_tok.load_bpe(find_input("bpe.json"))
    return _BPE

_bpe = bpe_tok_bpe()
PAD, UNK, BOS, EOS = (_bpe.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
VOCAB = len(_bpe.itos)

# ------------------------------------------------------------------------------------------------ data
def _text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))
    return x if isinstance(x, str) else None

def _items(obj):
    if isinstance(obj, dict) and isinstance(obj.get("data"), list):
        obj = obj["data"]
    return list(obj.items()) if isinstance(obj, dict) else [(None, x) for x in obj]

def read_queries(split):
    """-> list of {"id", "text", "gold": [candidate ids]}. Documented form: {query_id: {"text": [sentences],
    "relevant_candidates": [ids]}} (test: no relevant_candidates). Accepts a list of {"id", ...} as well."""
    obj = json.load(open(f"{DATA}/{split}.json", encoding="utf-8"))
    out = []
    for k, it in _items(obj):
        qid = k if k is not None else it.get("id", it.get("query_id", it.get("case_id")))
        text = _text(it.get("text", it.get("document")))
        if qid is None or text is None:
            raise ValueError(f"{split}.json: cannot read a query (keys: {list(it)[:8]}). Expected "
                             '{query_id: {"text": [...], "relevant_candidates": [...]}}')
        gold = [str(g) for g in (it.get("relevant_candidates") or []) if str(g).strip()]
        out.append({"id": str(qid), "text": text, "gold": list(dict.fromkeys(gold))})
    return out

def read_pool(split):
    """-> list of {"id", "text"} from <split>_candidates.json: {cand_id: {"text": [...]}} or {cand_id: [...]} or a
    list of {"id", "text"}"""
    obj = json.load(open(f"{DATA}/{split}_candidates.json", encoding="utf-8"))
    out = []
    for k, it in _items(obj):
        if isinstance(it, (str, list)):
            cid, text = k, _text(it)
        else:
            cid = k if k is not None else it.get("id", it.get("candidate_id", it.get("case_id")))
            text = _text(it.get("text", it.get("document")))
        if cid is None or text is None:
            raise ValueError(f"{split}_candidates.json: cannot read a candidate (got {type(it).__name__}).")
        out.append({"id": str(cid), "text": text})
    return out

# ------------------------------------------------------------------------------------------------ tokenisation cache
_ENC = None
def _init_worker():
    global _ENC
    _ENC = bpe_tok.FastEncoder(bpe_tok_bpe())

def _encode(text):
    t = np.asarray(_ENC.encode(text), dtype=np.int32)[:DOC_CAP]
    return t if len(t) else np.array([UNK], dtype=np.int32)

def build_cache(workers=None, limit=None):
    """Tokenises every query and candidate ONCE with your Phase-1 tokenizer (bpe_tok.FastEncoder, as in pretraining)
    and stores per split: all token ids concatenated + where each document starts (q_* for queries, c_* for the
    candidate pool) + ids and gold lists (meta.json)."""
    os.makedirs(CACHE, exist_ok=True)
    dtype = np.uint16 if VOCAB <= 65535 else np.int32
    for split in SPLITS:
        if os.path.exists(f"{CACHE}/{split}_meta.json"):
            continue
        qs, cs = read_queries(split)[:limit], read_pool(split)[:limit]
        with Pool(workers or os.cpu_count(), initializer=_init_worker) as pool:
            for kind, docs in (("q", qs), ("c", cs)):
                arrs = pool.map(_encode, [d["text"] for d in docs], chunksize=4)
                off = np.zeros(len(arrs) + 1, dtype=np.int64)
                off[1:] = np.cumsum([len(a) for a in arrs])
                np.save(f"{CACHE}/{split}_{kind}_ids.npy", np.concatenate(arrs).astype(dtype))
                np.save(f"{CACHE}/{split}_{kind}_off.npy", off)
                print(f"{split} {kind}: {len(docs)} documents, {off[-1]:,} tokens cached", flush=True)
        json.dump({"q_ids": [d["id"] for d in qs], "c_ids": [d["id"] for d in cs], "gold": [d["gold"] for d in qs]},
                  open(f"{CACHE}/{split}_meta.json", "w"))

class Split:
    """one cached split: query ids, candidate-pool ids, gold ids per query (train / dev), tokens of query i / candidate j"""
    def __init__(self, name):
        m = json.load(open(f"{CACHE}/{name}_meta.json"))
        self.name, self.q_ids, self.c_ids, self.gold = name, m["q_ids"], m["c_ids"], m["gold"]
        self.c_index = {c: j for j, c in enumerate(self.c_ids)}
        self.q_flat = np.load(f"{CACHE}/{name}_q_ids.npy", mmap_mode="r")
        self.q_off = np.load(f"{CACHE}/{name}_q_off.npy")
        self.c_flat = np.load(f"{CACHE}/{name}_c_ids.npy", mmap_mode="r")
        self.c_off = np.load(f"{CACHE}/{name}_c_off.npy")
        # gold candidates that are in the pool (indices); gold ids missing from the pool still count as misses
        self.gold_idx = [np.array(sorted({self.c_index[g] for g in gl if g in self.c_index}), dtype=np.int64) for gl in self.gold]
        self.n_gold = sum(len(g) for g in self.gold)
        self.self_pairs = [(i, self.c_index[q]) for i, q in enumerate(self.q_ids) if q in self.c_index]
    def n_q(self):
        return len(self.q_ids)
    def n_c(self):
        return len(self.c_ids)
    def q_tokens(self, i):
        return np.asarray(self.q_flat[self.q_off[i]:self.q_off[i + 1]], dtype=np.int32)
    def c_tokens(self, j):
        return np.asarray(self.c_flat[self.c_off[j]:self.c_off[j + 1]], dtype=np.int32)
    def q_lengths(self):
        return np.diff(self.q_off)
    def c_lengths(self):
        return np.diff(self.c_off)
    def mask_self(self, S):
        """a query that is itself in the candidate pool cannot retrieve itself (in place; S is a numpy / torch matrix)"""
        for i, j in self.self_pairs:
            S[i, j] = -float("inf")
        return S

# ------------------------------------------------------------------------------------------------ BM25 (lexical reference + hard negatives)
def _counts(flat, off, chunk=2000):
    import scipy.sparse as sp
    mats = []
    for a in range(0, len(off) - 1, chunk):
        b = min(a + chunk, len(off) - 1)
        rows = np.repeat(np.arange(b - a), np.diff(off[a:b + 1]))
        cols = np.asarray(flat[off[a]:off[b]], dtype=np.int64)
        mats.append(sp.csr_matrix((np.ones(len(cols), np.float32), (rows, cols)), shape=(b - a, VOCAB)))
    return sp.vstack(mats).tocsr()

_BM = {}
def bm25_matrix(split, k1=1.2, b=0.75):
    """(n_queries, n_candidates) BM25 scores of every query against its own split's pool, over Phase-1 BPE tokens.
    Query term weight 1 + ln(tf) (queries are whole cases). IDF from the pool only. Cached in memory."""
    key = (split.name, k1, b)
    if key in _BM:
        return _BM[key]
    C, Q = _counts(split.c_flat, split.c_off), _counts(split.q_flat, split.q_off)
    N = C.shape[0]
    df = np.bincount(C.indices, minlength=VOCAB)
    idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.diff(split.c_off).astype(np.float32)
    rows = np.repeat(np.arange(N), np.diff(C.indptr))
    C.data = C.data * (k1 + 1) / (C.data + k1 * (1 - b + b * dl[rows] / dl.mean())) * idf[C.indices]
    W = C.T.tocsr()
    Q.data = 1 + np.log(Q.data)
    out = np.zeros((Q.shape[0], N), dtype=np.float32)
    for a in range(0, Q.shape[0], 128):
        out[a:a + 128] = (Q[a:a + 128] @ W).toarray()
    _BM[key] = out
    return out

def build_aux(val_queries, hn_pool=100, minipool_k=30):
    """BM25-derived helper files (computed once): for every TRAIN query the `hn_pool` best-scoring candidates that are
    not relevant and not the query itself (hard-negative pool, best first); for the first `val_queries` DEV queries a
    small candidate set = their relevant candidates + the `minipool_k` best-scoring others (checkpoint selection)."""
    p = f"{CACHE}/aux_meta.json"
    want = {"val_queries": val_queries, "hn_pool": hn_pool, "minipool_k": minipool_k}
    if os.path.exists(p) and json.load(open(p)) == want:
        return
    tr, dv = Split("train"), Split("dev")
    S = bm25_matrix(tr).copy()
    tr.mask_self(S)
    for i, g in enumerate(tr.gold_idx):
        S[i, g] = -np.inf
    top = np.argsort(-S, axis=1)[:, :hn_pool]
    top = np.where(np.take_along_axis(S, top, 1) > -np.inf, top, -1).astype(np.int32)
    np.save(f"{CACHE}/train_hardneg.npy", top)
    S = bm25_matrix(dv).copy()
    dv.mask_self(S)
    mini = []
    for i in range(min(val_queries, dv.n_q())):
        g = dv.gold_idx[i]
        s = S[i].copy(); s[g] = -np.inf
        mini.append(sorted(set(g.tolist()) | set(np.argsort(-s)[:minipool_k].tolist())))
    json.dump(mini, open(f"{CACHE}/dev_minipool.json", "w"))
    json.dump(want, open(p, "w"))

# ------------------------------------------------------------------------------------------------ windows
def doc_windows(n, win):
    """non-overlapping windows covering the document; the last one is aligned to the end. A document shorter than
    `win` is one window cut to a multiple of LEN_BUCKET tokens (your models have no padding mask, so tokens that
    share a forward pass must have equal lengths)."""
    if n <= win:
        return [(0, n if n < LEN_BUCKET else n - n % LEN_BUCKET)]
    starts = list(range(0, n - win, win)) + [n - win]
    return [(s, s + win) for s in sorted(set(starts))]

def pick_windows(ws, cap):
    """indices of at most `cap` windows, evenly spread (first and last always included)"""
    if len(ws) <= cap:
        return list(range(len(ws)))
    return sorted(set(np.linspace(0, len(ws) - 1, cap).round().astype(int).tolist()))

# ------------------------------------------------------------------------------------------------ model wrappers
class Backbone(nn.Module):
    """Wraps one of YOUR Phase-1 models so that backbone(ids) returns hidden states (B, T, d_model) instead of
    next-token scores (output layer replaced by Identity). encdec: the ENCODER output. prefix: the whole window is the
    (bidirectional) prefix. Decoder / swa / moe: causal."""
    def __init__(self, arch, lm):
        super().__init__()
        self.arch, self.lm = arch, lm
        self.kind = getattr(lm, "kind", "causal")
        self.d_model = lm.tok_emb.embedding_dim
        lm.head = nn.Identity()
        if arch == "encdec":
            del lm.dec_layers, lm.dec_norm, lm.tgt_pos_emb

    def forward(self, ids):
        T = ids.size(1)
        if self.arch == "encdec":
            lm = self.lm
            h = lm.tok_emb(ids) + lm.src_pos_emb(torch.arange(T, device=ids.device))
            for layer in lm.enc_layers:
                h = layer(h)
            return lm.enc_norm(h)
        if self.kind == "prefix":
            return self.lm(ids, prefix_len=T)
        return self.lm(ids)

class RetModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

TINY = {   # used only by the CPU smoke test (random weights, no checkpoint)
    "decoder":   dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "prefix":    dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "encdec":    dict(d_model=32, n_enc_layers=2, n_dec_layers=2, n_heads=4, max_len=192, max_tgt_len=64, dropout=0.0),
    "swa":       dict(d_model=32, n_layers=2, n_heads=4, window=16, max_len=256, dropout=0.0),
    "diffusion": dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "moe":       dict(d_model=32, n_layers=2, n_heads=4, d_ff=64, n_experts=4, max_len=256, dropout=0.0),
}

def build_lm(arch, cfg):
    import models, models_x
    return (models_x if arch in ("diffusion", "moe") else models).build_model(arch, vocab_size=VOCAB, **cfg)

def load_holder(arch, ckpt_pattern, smoke=False):
    if smoke:
        lm = build_lm(arch, TINY[arch])
    else:
        ck = torch.load(find_input(ckpt_pattern), map_location="cpu", weights_only=False)
        assert ck.get("arch", arch) == arch, f"checkpoint {ckpt_pattern} is a '{ck.get('arch')}' model, not '{arch}'"
        lm = build_lm(arch, ck["cfg"])
        lm.load_state_dict(ck["model"])
    return Backbone(arch, lm)

def build_model(e, checkpoints, M, smoke=False):
    """one experiment's model: its Phase-1 backbone (+ the first `student_layers` layers if a student), frozen, then
    YOUR method makes the right parts trainable. method "none" = untouched Phase-1 model (no fine-tuning)."""
    holder = load_holder(e["arch"], checkpoints[e["arch"]], smoke=smoke)
    if e.get("student_layers"):
        holder = M.make_student(holder, e["student_layers"])
    model = RetModel(holder)
    for p in model.parameters():
        p.requires_grad_(False)
    if e["method"] != "none":
        M.METHODS[e["method"]](model, **e["method_cfg"])
    return model

def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

def load_trained(model, state):
    res = model.load_state_dict(state, strict=False)
    assert not res.unexpected_keys, f"checkpoint has parameters the model does not: {res.unexpected_keys[:3]}"

def iter_adaptable(module, skip=("head",)):
    for name, mod in module.named_modules():
        if any(s in name.split(".") for s in skip):
            continue
        if isinstance(mod, nn.MultiheadAttention) and getattr(mod, "in_proj_weight", None) is not None:
            yield name, mod, "in_proj_weight", mod.in_proj_weight
        elif isinstance(mod, nn.Linear):
            yield name, mod, "weight", mod.weight

def layer_stack(backbone):
    lm = backbone.lm
    if backbone.arch == "encdec":
        return "enc_layers", lm.enc_layers, lm.enc_norm
    if hasattr(lm, "blocks"):
        return "blocks", lm.blocks, lm.norm
    return "layers", lm.layers, lm.norm

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    return total, sum(p.numel() for p in model.parameters() if p.requires_grad)

def merge_adapters(model):
    """bakes every parametrisation (LoRA, LoRMA, IA3, ...) into its weight — same outputs, faster inference"""
    for mod in model.modules():
        if parametrize.is_parametrized(mod):
            for name in list(mod.parametrizations.keys()):
                parametrize.remove_parametrizations(mod, name, leave_parametrized=True)

# ------------------------------------------------------------------------------------------------ encoding
def n_prompt_of(e):
    return int(e["method_cfg"].get("n_tokens", 20)) if e["method"] == "prompt" else 0

def window_size(lm, n_prompt, win_tokens):
    w = min(win_tokens, lm.max_len - n_prompt)
    assert w >= 32, f"no room for a window: max_len {lm.max_len}, prompt {n_prompt}"
    return w

def pool_tokens(h, how):
    if how == "mean":
        return h.mean(1)
    if how == "max":
        return h.max(1).values
    if how == "last":
        return h[:, -1]
    raise ValueError(f"token_pool must be mean / max / last, got {how!r}")

def encode(model, seqs, n_prompt, device, pool="mean", batch_tokens=24000):
    """window token arrays -> (N, d_model) float32, L2-normalised, in input order. [pad]*n_prompt is put in front of every
    window (prompt tuning); only the document positions are pooled. Windows of equal length share a forward pass.
    Differentiable (training calls this with grad enabled)."""
    cuda = str(device).startswith("cuda")
    by = {}
    for j, s in enumerate(seqs):
        by.setdefault(len(s), []).append(j)
    pieces, order = [], []
    for T, js in by.items():
        bs = max(1, batch_tokens // (T + n_prompt))
        for a in range(0, len(js), bs):
            jj = js[a:a + bs]
            x = torch.tensor(np.stack([np.concatenate([np.full(n_prompt, PAD, np.int32), seqs[j]]) for j in jj]),
                             dtype=torch.long, device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
                h = model.backbone(x)[:, n_prompt:]
            pieces.append(F.normalize(pool_tokens(h.float(), pool), dim=-1))
            order += jj
    E = torch.cat(pieces)
    inv = torch.empty(len(order), dtype=torch.long, device=E.device)
    inv[torch.tensor(order, device=E.device)] = torch.arange(len(order), device=E.device)
    return E[inv]

# ------------------------------------------------------------------------------------------------ metric + files
def micro_f1(top, gold_idx, n_gold):
    """Micro-F1@7 / precision / recall exactly as evaluate_phase7.py: `top` (n_queries, 7) candidate indices, gold_idx
    = per-query arrays of relevant candidate indices, n_gold = number of ALL relevant ids (also those outside the pool)"""
    tp = sum(len(set(map(int, r)) & set(map(int, g))) for r, g in zip(top, gold_idx))
    n = len(top) * K
    fp, fn = n - tp, n_gold - tp
    return (2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0), (tp / n if n else 0.0), (tp / n_gold if n_gold else 0.0)

def find_runs(filename, sub="pcr_runs"):
    """{experiment: folder} for every '.../<sub>/<experiment>/<filename>' under /kaggle/working and /kaggle/input.
    Found more than once: /kaggle/working wins, then the newest file."""
    hits = {}
    for rank, root in enumerate((WORK, INPUT_ROOT)):
        for p in glob.glob(f"{root}/**/{sub}/*/{filename}", recursive=True):
            folder = os.path.dirname(p)
            exp, key = os.path.basename(folder), (rank, -os.path.getmtime(p))
            if exp not in hits or key < hits[exp][0]:
                hits[exp] = (key, folder)
    return {e: f for e, (_, f) in sorted(hits.items())}

def prediction_dict(q_ids, c_ids, top):
    return {q: [c_ids[int(j)] for j in row] for q, row in zip(q_ids, top)}

def write_submission(pred, path_json, path_zip=None):
    json.dump(pred, open(path_json, "w", encoding="utf-8"), ensure_ascii=False)
    if path_zip:
        import zipfile
        with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(path_json, arcname="prediction.json")

## N5. Your Phase-1 model code (unchanged copies of your `models.py` and `models_x.py`)

In [ ]:
%%writefile models.py
# =====================================================================================================
#  CS-781 Phase 1 — four language models                      
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
#  PART 1 — MASKS
# =====================================================================================================

def causal_mask(T, device):
    mask = torch.ones(T, T, dtype=torch.bool, device=device).triu(diagonal=1)
    return mask


def prefix_mask(T, P, device):
    mask = causal_mask(T, device)
    mask[:, :P] = False
    return mask


def alibi_window_bias(T, n_heads, window, device):
    i = torch.arange(T, device=device)[:, None]
    j = torch.arange(T, device=device)[None, :]

    dist = i - j

    slopes = torch.tensor(
        [2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)],
        device=device,
        dtype=torch.float32,
    )[:, None, None]

    bias = -slopes * dist

    bias = bias.masked_fill(
        (dist < 0) | (dist >= window),
        float("-inf")
    )

    return bias


# =====================================================================================================
#  PART 2 — SHARED HELPERS
# =====================================================================================================

def make_layers(n_layers, d_model, n_heads, dropout):

    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
#  MODEL 1 — DECODER-ONLY (GPT-style)
# =====================================================================================================

class DecoderLM(nn.Module):
    kind = "causal"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = causal_mask(T, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 2 — PREFIX-LM
# =====================================================================================================

class PrefixLM(nn.Module):
    kind = "prefix"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, prefix_len):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = prefix_mask(T, prefix_len, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 3 — ENCODER-DECODER
# =====================================================================================================

class EncDecLM(nn.Module):
    kind = "encdec"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_enc_layers,
        n_dec_layers,
        n_heads,
        max_len,
        max_tgt_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.max_tgt_len = max_len, max_tgt_len

        # Shared token embedding
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # Separate position embeddings
        self.src_pos_emb = nn.Embedding(max_len, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len, d_model)

        # Encoder
        self.enc_layers = make_layers(
            n_enc_layers,
            d_model,
            n_heads,
            dropout
        )

        self.enc_norm = nn.LayerNorm(d_model)

        # Decoder
        self.dec_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_dec_layers)
        ])

        self.dec_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, src, tgt_in):
        S, U = src.size(1), tgt_in.size(1)

        # -------------------------
        # ENCODER
        # -------------------------

        src_pos = torch.arange(S, device=src.device)

        mem = self.tok_emb(src) + self.src_pos_emb(src_pos)

        for layer in self.enc_layers:
            mem = layer(mem)

        mem = self.enc_norm(mem)

        # -------------------------
        # DECODER
        # -------------------------

        tgt_pos = torch.arange(U, device=tgt_in.device)

        y = self.tok_emb(tgt_in) + self.tgt_pos_emb(tgt_pos)

        tgt_mask = causal_mask(U, tgt_in.device)

        for layer in self.dec_layers:
            y = layer(
                y,
                mem,
                tgt_mask=tgt_mask
            )

        return self.head(self.dec_norm(y))


# =====================================================================================================
#  MODEL 4 — EFFICIENT: SLIDING-WINDOW ATTENTION + ALiBi
# =====================================================================================================

class SlidingWindowLM(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        window,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.n_heads, self.window = max_len, n_heads, window

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings only
        x = self.tok_emb(idx)

        # ALiBi + sliding-window bias
        bias = alibi_window_bias(
            T,
            self.n_heads,
            self.window,
            idx.device
        )

        # (n_heads, T, T)
        # -> (B * n_heads, T, T)
        bias = bias.repeat(B, 1, 1)

        for layer in self.layers:
            x = layer(
                x,
                src_mask=bias
            )

        return self.head(self.norm(x))


# =====================================================================================================
#  FACTORY — do not change.
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "decoder": DecoderLM,
        "prefix": PrefixLM,
        "encdec": EncDecLM,
        "swa": SlidingWindowLM
    }

    return classes[arch](vocab_size, **cfg)

In [ ]:
%%writefile models_x.py

# =====================================================================================================
# CS-781 Phase 1 — two extra architectures
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
# PART 0 — HELPERS
# =====================================================================================================

def causal_mask(T, device):
    return torch.ones(
        T, T,
        dtype=torch.bool,
        device=device
    ).triu(diagonal=1)


def make_layers(n_layers, d_model, n_heads, dropout):
    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(
            module.weight,
            mean=0.0,
            std=0.02
        )

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
# MODEL 5 — LexDiffusionLM
# =====================================================================================================

class LexDiffusionLM(nn.Module):
    kind = "diffusion"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer layers
        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Fully bidirectional attention.
        # NO MASK.
        for layer in self.layers:
            x = layer(x)

        # Final normalization + output
        x = self.norm(x)

        return self.head(x)


# =====================================================================================================
# MODEL 6 — LexMoE
# =====================================================================================================

class MoEFeedForward(nn.Module):
    """
    Top-1 Mixture-of-Experts feed-forward network.

    Each token is routed to exactly one expert.
    """

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        self.n_experts = n_experts

        # Filled during forward()
        self.aux_loss = None
        self.last_choice = None

        # Router
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False
        )

        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
                nn.Dropout(dropout),
            )
            for _ in range(n_experts)
        ])


    def forward(self, x):
        B, T, D = x.shape

        # Flatten tokens
        flat = x.reshape(
            B * T,
            D
        )

        # Router probabilities
        router_logits = self.router(flat)

        # Float32 router softmax for numerical stability
        probs = torch.softmax(
            router_logits.float(),
            dim=-1
        )

        # Best expert for each token
        top_p, top_i = probs.max(
            dim=-1
        )

        # Output buffer
        out = flat.new_zeros(
            flat.shape
        )

        # ------------------------------------------------
        # Route tokens to experts
        # ------------------------------------------------

        for e in range(self.n_experts):

            sel = (top_i == e)

            if sel.any():

                expert_out = self.experts[e](
                    flat[sel]
                )

                expert_out = (
                    expert_out
                    * top_p[sel][:, None]
                )

                out[sel] = expert_out.to(
                    out.dtype
                )


        # ------------------------------------------------
        # Switch load-balancing loss
        # ------------------------------------------------

        counts = torch.bincount(
            top_i,
            minlength=self.n_experts
        ).float()

        frac_e = counts / flat.shape[0]

        prob_e = probs.mean(
            dim=0
        )

        self.aux_loss = (
            self.n_experts
            * torch.sum(frac_e * prob_e)
        )


        # Store routing decisions
        self.last_choice = top_i.reshape(
            B,
            T
        )


        # Restore original shape
        return out.reshape(
            B,
            T,
            D
        )


# =====================================================================================================
# MoE BLOCK
# =====================================================================================================

class MoEBlock(nn.Module):
    """
    Pre-norm Transformer block:

        x -> LayerNorm -> Attention -> residual
          -> LayerNorm -> MoE      -> residual
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        # Pre-norm attention
        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.drop = nn.Dropout(
            dropout
        )

        # Pre-norm MoE
        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.moe = MoEFeedForward(
            d_model,
            d_ff,
            n_experts,
            dropout
        )


    def forward(self, x, mask):

        # ------------------------------------------------
        # Attention
        # ------------------------------------------------

        h = self.norm1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=mask,
            need_weights=False
        )

        x = x + self.drop(
            attn_out
        )


        # ------------------------------------------------
        # MoE feed-forward
        # ------------------------------------------------

        moe_out = self.moe(
            self.norm2(x)
        )

        x = x + moe_out

        return x


# =====================================================================================================
# LexMoE
# =====================================================================================================

class LexMoE(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        n_experts,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Filled during forward()
        self.aux_loss = None

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # MoE Transformer blocks
        self.blocks = nn.ModuleList([
            MoEBlock(
                d_model,
                n_heads,
                d_ff,
                n_experts,
                dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(
            d_model
        )

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Causal mask
        mask = causal_mask(
            T,
            idx.device
        )

        # Store auxiliary losses
        aux_losses = []

        # Transformer blocks
        for block in self.blocks:

            x = block(
                x,
                mask
            )

            aux_losses.append(
                block.moe.aux_loss
            )

        # Average auxiliary loss across layers
        self.aux_loss = torch.stack(
            aux_losses
        ).mean()

        # Final normalization
        x = self.norm(x)

        # Vocabulary logits
        return self.head(x)


# =====================================================================================================
# FACTORY — DO NOT CHANGE
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "diffusion": LexDiffusionLM,
        "moe": LexMoE
    }

    return classes[arch](
        vocab_size,
        **cfg
    )

## N6. `pcr_methods.py` — YOUR CODE (unchanged from Phase 6) + two functions to write
`method_full`, `method_partial`, `method_bitfit`, `method_lora`, `method_lorma`, `method_rslora`, `method_ia3`,
`method_prompt`, `make_student` work as they are: the pipeline hands them your Phase-1 model as `model.backbone.lm`
(without its output layer — the model is a window encoder here). `TokenClassifier`, `kd_loss` and `kd_loss_multilabel`
are not used this phase. **New — section 9 at the bottom:** `info_nce_loss` (the contrastive objective) and
`kd_scores_loss` (score-distribution distillation). The steps are written out there; N7b stops the session if either is
still missing.

In [ ]:
%%writefile pcr_methods.py
# =====================================================================================================
#  CS-781 Phase 7 (copied from Phase 6) — YOUR CODE: the NER head, the fine-tuning methods, and knowledge distillation
#                                                                AUTHOR: <your name>   (your own code)
# =====================================================================================================

import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import parametrize
import pcr_common as L


# =====================================================================================================
#  1. THE NER HEAD
# =====================================================================================================

class TokenClassifier(nn.Module):

    def __init__(self, backbone, n_tags, dropout):
        super().__init__()

        # Keep backbone
        self.backbone = backbone

        # Dropout before classifier
        self.dropout = nn.Dropout(dropout)

        # Token-level classification head
        self.head = nn.Linear(
            backbone.d_model,
            n_tags
        )


    def forward(self, ids):

        # Backbone hidden states
        h = self.backbone(ids)

        # Classification
        return self.head(
            self.dropout(h)
        )


# =====================================================================================================
#  2. SIMPLE METHODS: full, partial, BitFit
# =====================================================================================================

def method_full(model, **cfg):
    """Full fine-tuning: every parameter of the backbone trains."""

    for p in model.backbone.parameters():
        p.requires_grad_(True)


def method_partial(model, n_last=2, **cfg):
    """
    Partial / layer-wise fine-tuning:
    only the LAST n_last transformer layers + final LayerNorm train.
    """

    # Get the relevant layer stack
    name, layers, final_norm = L.layer_stack(
        model.backbone
    )

    # Train only the last n_last layers
    for layer in layers[-n_last:]:
        for p in layer.parameters():
            p.requires_grad_(True)

    # Train final LayerNorm
    for p in final_norm.parameters():
        p.requires_grad_(True)


def method_bitfit(model, **cfg):
    """
    BitFit:
    only bias parameters of the backbone train.
    """

    for name, p in model.backbone.named_parameters():

        if name.endswith("bias"):
            p.requires_grad_(True)


# =====================================================================================================
#  3. LoRA
# =====================================================================================================

class LoRAParam(nn.Module):
    """
    Additive low-rank update:

        W' = W + (alpha / r) * B @ A
    """

    def __init__(
        self,
        d_out,
        d_in,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.empty(r, d_in)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=math.sqrt(5)
        )

        # B: zero initialization
        # This guarantees W' = W initially.
        self.B = nn.Parameter(
            torch.zeros(d_out, r)
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        return W + self.scale * (
            self.B @ self.A
        )


def method_lora(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRAParam(
                d_out,
                d_in,
                r,
                alpha
            )
        )


# =====================================================================================================
#  4. LoRMA
# =====================================================================================================

class LoRMAParam(nn.Module):
    """
    LoRMA+ multiplicative low-rank update:

        W' = W + (alpha / r) * B @ (A @ W)

    where:

        A : (r, d_out)
        B : (d_out, r)
    """

    def __init__(
        self,
        d_out,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.randn(
                r,
                d_out
            ) * 0.02
        )

        # B: zero initialization
        self.B = nn.Parameter(
            torch.zeros(
                d_out,
                r
            )
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        # First compute A @ W
        AW = self.A @ W

        # Then B @ (A @ W)
        update = self.B @ AW

        return W + self.scale * update


def method_lorma(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRMAParam(
                d_out,
                r,
                alpha
            )
        )


# =====================================================================================================
#  5. KNOWLEDGE DISTILLATION
# =====================================================================================================

def kd_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.5
):
    """
    Combined hard-label cross entropy and
    teacher-student KL divergence.
    """

    # Only real labelled tokens
    mask = labels != -100

    s = student_logits[mask]
    t = teacher_logits[mask]
    y = labels[mask]

    # --------------------------------------------------------
    # Hard-label loss
    # --------------------------------------------------------

    hard = F.cross_entropy(
        s,
        y
    )

    # --------------------------------------------------------
    # Soft teacher/student distributions
    # --------------------------------------------------------

    soft = F.kl_div(
        F.log_softmax(
            s / temperature,
            dim=-1
        ),
        F.softmax(
            t / temperature,
            dim=-1
        ),
        reduction="batchmean"
    )

    # Hinton temperature scaling
    soft = soft * (
        temperature * temperature
    )

    # Combined loss
    return (
        alpha * hard
        + (1.0 - alpha) * soft
    )


def make_student(
    backbone,
    keep_layers
):
    """
    Make a smaller copy of a Phase-1 backbone,
    keeping only its first keep_layers layers.
    """

    # Deep copy so original backbone is untouched
    small = copy.deepcopy(
        backbone
    )

    # Find the layer stack
    name, layers, _ = L.layer_stack(
        small
    )

    # Keep only first keep_layers
    setattr(
        small.lm,
        name,
        nn.ModuleList(
            list(layers)[:keep_layers]
        )
    )

    return small


# =====================================================================================================
#  METHOD TABLE
# =====================================================================================================

METHODS = {
    "full": method_full,
    "partial": method_partial,
    "bitfit": method_bitfit,
    "lora": method_lora,
    "lorma": method_lorma,
}


# =====================================================================================================
#  6. YOUR PHASE-3 ADDITIONS (rsLoRA, IA³) — unchanged. Note for Phase 4: the §10.2.1 list is Full fine-tuning,
#     Partial/Layer-wise fine-tuning, Layer freezing/unfreezing, BitFit, LORMA, AdaLoRA, Adapters, Knowledge
#     Distillation. rsLoRA, IA³ and LoRA are NOT on it, so they are optional extras this phase. You already have
#     full, partial (method_partial above), BitFit, LoRMA and distillation — no new method is needed.
# =====================================================================================================

def method_rslora(model, r=8, alpha=16, skip=("head",), **cfg):
    """Rank-stabilised LoRA: use alpha / sqrt(r) as the update scale."""
    if r <= 0:
        raise ValueError("r must be a positive integer")

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out, d_in = W.shape
        adapter = LoRAParam(d_out, d_in, r, alpha)
        adapter.scale = alpha / math.sqrt(r)
        parametrize.register_parametrization(owner, pname, adapter)


# -----------------------------------------------------------------------------------------------------
#  B. IA³ — "Infused Adapter by Inhibiting and Amplifying Inner Activations" (Liu et al. 2022)
# -----------------------------------------------------------------------------------------------------
#  In plain English:
#    IA³ adds no matrices at all. Every adapted weight gets ONE learned vector, with one number per output
#    row of the weight. Each row of the weight is multiplied by its number. The vector starts as all ones, so
#    at the start the model is unchanged (the same idea as B starting at zero in LoRA). Only these vectors and
#    the head are trained, so IA³ trains far fewer parameters than LoRA.
#
#  What you have to write — two pieces:
#
#    Piece 1 — a small class, written just above method_ia3, shaped like your LoRAParam:
#      1. It inherits from nn.Module and its __init__ receives d_out (the number of output rows of the weight).
#      2. In __init__, after super().__init__(), create one trainable vector of length d_out filled with ones
#         (an nn.Parameter wrapped around torch.ones of size d_out). Call it something like "l".
#      3. Its forward receives the original weight W (shape d_out x d_in) and returns W with every row multiplied
#         by the matching number of the vector. To do that, turn the vector into a column of shape d_out x 1
#         (index it with [:, None]) and multiply it with W — PyTorch repeats the column across all d_in columns.
#
#    Piece 2 — the function method_ia3:
#      4. Go through every adaptable weight with the same loop as method_lora.
#      5. Read the weight's number of output rows (the first number of W.shape).
#      6. Create your Piece-1 object with that number and attach it with parametrize.register_parametrization,
#         exactly like method_lora attaches LoRAParam.
#      7. Delete the "raise NotImplementedError" line.
#
#    For your report: the paper rescales only keys, values and the feed-forward hidden layer. Rescaling every
#    adaptable weight (what steps 4-6 do) is a simpler variant — say which one you used.

class IA3Param(nn.Module):
    """Row-wise IA³ scaling for a weight matrix."""

    def __init__(self, d_out):
        super().__init__()
        self.l = nn.Parameter(torch.ones(d_out))

    def forward(self, W):
        return self.l[:, None] * W


def method_ia3(model, skip=("head",), **cfg):
    """Apply trainable row-wise scaling to every adaptable backbone weight."""
    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out = W.shape[0]
        parametrize.register_parametrization(
            owner,
            pname,
            IA3Param(d_out),
        )


# Do not change this line: it makes "rslora" and "ia3" usable as "method" in N3.
METHODS.update({"rslora": method_rslora, "ia3": method_ia3})


# =====================================================================================================
#  7. NEW FOR PHASE 5 — distillation loss for MULTI-LABEL outputs. YOU write this function (about 6 lines).
# =====================================================================================================
#  Why a new one: your kd_loss above is for single-label classification — a softmax over the classes, and
#  cross_entropy against ONE class index per token. In LSI every statute is its own yes/no decision (a document can
#  have several), so the teacher's knowledge is K independent probabilities, one sigmoid per statute.
#
#  What train_ls.py passes in (already pooled to ONE row per document):
#    student_logits  (B, K)  the student's statute scores        teacher_logits (B, K)  the teacher's
#    targets         (B, K)  the true statutes, 1.0 / 0.0         temperature, alpha     from "kd" in N3
#
#  Steps:
#    1. hard loss: F.binary_cross_entropy_with_logits(student_logits, targets)
#    2. the teacher's soft targets: torch.sigmoid(teacher_logits / temperature)
#    3. soft loss: F.binary_cross_entropy_with_logits(student_logits / temperature, <the soft targets of step 2>),
#       multiplied by temperature * temperature (the same Hinton scaling as in your kd_loss)
#    4. return alpha * hard + (1 - alpha) * soft
#    5. delete the "raise NotImplementedError" line.
#  For your report: this is the per-statute (binary) form of Hinton et al.'s distillation.

def kd_loss_multilabel(student_logits, teacher_logits, targets, temperature=2.0, alpha=0.5):
    hard = F.binary_cross_entropy_with_logits(student_logits, targets)
    soft_targets = torch.sigmoid(teacher_logits.detach() / temperature)
    soft = F.binary_cross_entropy_with_logits(
        student_logits / temperature, soft_targets
    ) * (temperature * temperature)
    return alpha * hard + (1.0 - alpha) * soft


# =====================================================================================================
#  8. NEW FOR PHASE 6 — Prompt Tuning (Lester et al. 2021). YOU write this (about 15 lines).
# =====================================================================================================
#  Why: the §12.2.1 list is Houlsby / Pfeiffer Adapters, Prefix Tuning, Prompt Tuning, LoRA, QLoRA, AdaLoRA, LoRMA,
#  Knowledge Distillation. You already have LoRA, LoRMA and distillation; you need ONE more listed technique.
#  Full fine-tuning, partial, BitFit, rsLoRA and IA³ are not on the list.
#
#  The idea: the whole Phase-1 model stays frozen. Only n_tokens learned vectors (a "soft prompt", d_model numbers
#  each) are trained. They are put in front of every input, and the model learns to read them as an instruction
#  ("summarise this").
#
#  What the pipeline already does for you: when an experiment's method is "prompt", every input sequence (encdec:
#  the encoder input) starts with n_tokens <pad> placeholder tokens. Your job is to swap the embeddings of those
#  placeholders for your learned vectors. A forward hook on the token-embedding layer does that: PyTorch calls
#  hook(module, inputs, output) after every tok_emb(ids) call, and if the hook returns a tensor, it replaces the output.
#
#  Piece 1 — class SoftPrompt(nn.Module), written just above method_prompt:
#    1. __init__(self, tok_emb, n_tokens, pad_id): after super().__init__(), pick n_tokens random row numbers of the
#       embedding table (torch.randint(0, tok_emb.num_embeddings, (n_tokens,))) and make
#       self.prompt = nn.Parameter(tok_emb.weight[those rows].detach().clone())       -> shape (n_tokens, d_model)
#       (starting from real word embeddings trains better than random numbers). Also store n_tokens and pad_id.
#    2. def hook(self, module, inputs, output):   inputs[0] = the token ids (B, T); output = their embeddings (B, T, d)
#       a. n = self.n_tokens. If T <= n, return output unchanged (generation feeds one new token at a time).
#       b. rows = (ids[:, :n] == self.pad_id).all(dim=1)   -> which rows start with the placeholders (the encdec
#          decoder's input starts with <s>, so it is left alone)
#       c. if no row is True, return output unchanged.
#       d. output = output.clone(); output[rows, :n] = self.prompt.to(output.dtype); return output
#
#  Piece 2 — method_prompt(model, n_tokens=20, **cfg):
#    3. sp = SoftPrompt(model.backbone.lm.tok_emb, n_tokens, L.PAD)
#    4. model.soft_prompt = sp            (makes it part of the model: its parameter trains and is saved)
#    5. model.backbone.lm.tok_emb.register_forward_hook(sp.hook)
#    6. delete the "raise NotImplementedError" line.
#  Everything else stays frozen (the training script freezes the model before calling your method).
#  For your report: Lester et al. used ~20-100 prompt tokens on much larger models; say how many you used.

class SoftPrompt(nn.Module):
    """Learnable soft-prompt embeddings inserted at the start of input sequences."""

    def __init__(self, tok_emb, n_tokens, pad_id):
        super().__init__()

        if n_tokens <= 0:
            raise ValueError("n_tokens must be a positive integer")

        # Initialise from existing token embeddings rather than random values.
        rows = torch.randint(
            low=0,
            high=tok_emb.num_embeddings,
            size=(n_tokens,)
        )
        self.prompt = nn.Parameter(
            tok_emb.weight[rows].detach().clone()
        )
        self.n_tokens = n_tokens
        self.pad_id = pad_id

    def hook(self, module, inputs, output):
        # Generation may call the embedding layer with only one new token.
        n = self.n_tokens
        if output.shape[1] <= n:
            return output

        ids = inputs[0]
        # Replace only sequences whose first n tokens are all PAD placeholders.
        rows = (ids[:, :n] == self.pad_id).all(dim=1)
        if not rows.any():
            return output

        output = output.clone()
        prompt = self.prompt.to(device=output.device, dtype=output.dtype)
        output[rows, :n] = prompt
        return output


def method_prompt(model, n_tokens=20, **cfg):
    """Freeze-compatible prompt tuning using learned input embeddings."""
    sp = SoftPrompt(
        model.backbone.lm.tok_emb,
        n_tokens,
        L.PAD
    )
    # Register the prompt as part of the model so it is optimised and saved.
    model.soft_prompt = sp
    # Replace the PAD placeholder embeddings on each token-embedding lookup.
    model.backbone.lm.tok_emb.register_forward_hook(sp.hook)


# Do not change this line: it makes "prompt" usable as "method" in N3.
METHODS.update({"prompt": method_prompt})


# =====================================================================================================
#  9. NEW FOR PHASE 7 — the two retrieval losses. YOU write both (about 4 lines each).
# =====================================================================================================
#  Every other method above (full, partial, BitFit, LoRA, LoRMA, rsLoRA, IA³, prompt tuning, make_student) works
#  unchanged: the pipeline hands each one your Phase-1 model as model.backbone.lm and trains it as a window ENCODER —
#  a window of the case goes in, the hidden states are averaged over the tokens and normalised to length 1, so the
#  similarity of two windows is their dot product (= cosine). kd_loss (token-level) and kd_loss_multilabel are not used.
#
#  What train_pcr.py hands to your two functions, per training step (B queries, M candidate windows):
#    q            (B, d)  normalised query-window vectors
#    c            (M, d)  normalised candidate-window vectors: the FIRST B are the positives (row i = a relevant case of
#                         query i), the rest are negatives (BM25 hard negatives and random cases from the pool)
#    pos_index    (B,)    the column of each query's positive = tensor([0, 1, ..., B-1])
#    ignore       (B, M)  True where candidate m is ANOTHER relevant case of query i (it must not be treated as a negative)
#    temperature          from N3 ("temperature" for the contrastive loss, "kd"/"temperature" for distillation)
#
#  ---- Piece 1: info_nce_loss(q, c, pos_index, temperature, ignore=None) -> a scalar --------------------------
#    1. scores = q @ c.T / temperature                       (B, M): the query-vs-candidate similarity matrix
#    2. if ignore is not None: scores = scores.masked_fill(ignore, -1e4)    (those columns drop out of the softmax)
#    3. return F.cross_entropy(scores, pos_index)            "pick your own positive out of all M candidates"
#    4. delete the "raise NotImplementedError" line.
#    This is InfoNCE (Oord et al. 2018) with in-batch + hard negatives; the temperature sets how sharply the model must
#    separate the positive from its hardest negatives. For your report: with M = B + B*hard_negs candidates, every
#    query sees its own hard negatives AND the other queries' candidates as in-batch negatives.
#
#  ---- Piece 2: kd_scores_loss(student_scores, teacher_scores, temperature, ignore=None) -> a scalar ------------
#    student_scores / teacher_scores  (B, M)  raw cosine similarities (NOT divided by a temperature yet) of the same
#    queries and candidates from the student and from the teacher.
#    1. if ignore is not None: fill BOTH with -1e4 where ignore is True (masked_fill, as above)
#    2. log_p = F.log_softmax(student_scores / temperature, dim=-1)       the student's distribution over candidates
#       p_t   = F.softmax(teacher_scores.detach() / temperature, dim=-1)  the teacher's distribution
#    3. return F.kl_div(log_p, p_t, reduction="batchmean") * temperature * temperature
#    4. delete the "raise NotImplementedError" line.
#    The student learns the teacher's whole ranking over the candidates (which ones are close, how close), not only
#    which one is the positive. train_pcr.py combines it as alpha * info_nce + (1 - alpha) * kd (alpha in "kd", N3).
#    For your report: this is score-distribution distillation, one of the signals §13.2.2 lists.

def info_nce_loss(q, c, pos_index, temperature, ignore=None):
    scores = (q @ c.T) / temperature

    if ignore is not None:
        scores = scores.masked_fill(ignore, -1e4)

    return F.cross_entropy(scores, pos_index)


def kd_scores_loss(student_scores, teacher_scores, temperature, ignore=None):
    if ignore is not None:
        student_scores = student_scores.masked_fill(ignore, -1e4)
        teacher_scores = teacher_scores.masked_fill(ignore, -1e4)

    log_p = F.log_softmax(student_scores / temperature, dim=-1)
    p_t = F.softmax(teacher_scores.detach() / temperature, dim=-1)

    return F.kl_div(log_p, p_t, reduction="batchmean") * (temperature ** 2)

## N7. Scripts: training (one experiment), embedding (one run), the lab, the official Phase-7 evaluator

In [ ]:
%%writefile train_pcr.py
"""Runs ONE experiment from run_config_pcr.json: your Phase-1 model as a window encoder + your fine-tuning method,
trained contrastively (YOUR info_nce_loss) on (query window, relevant-candidate window, hard-negative windows); a
distillation student also matches a teacher's query-candidate similarity distribution (YOUR kd_scores_loss).
Checkpoint selection: Micro-F1@7 on a small DEV re-ranking task (first val_queries dev queries, candidates = their
relevant ones + the minipool_k best BM25 others, first window of each) every eval_minutes; the best is kept.
   python train_pcr.py --exp NAME --device cuda:0        (real run)
   python train_pcr.py --exp NAME --device cpu --smoke   (tiny random model, few documents: checks the code)"""
import argparse, json, os, random, time
import numpy as np, torch
import pcr_common as L
import pcr_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--exp", required=True); ap.add_argument("--device", default="cuda:0")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
dev = args.device
cuda = dev.startswith("cuda")
if cuda:
    torch.cuda.set_device(dev)

C = json.load(open(f"{L.WORK}/run_config_pcr.json"))
def exp_cfg(name):
    if name in C["experiments"]:
        e = {**C["defaults"], **C["experiments"][name], "name": name}
        return {**e, "win_tokens": 64} if args.smoke else e
    raise KeyError(f"experiment '{name}' is not in EXPERIMENTS (cell N3)")
E = exp_cfg(args.exp)
RUNS = f"{L.WORK}/{'pcr_runs_smoke' if args.smoke else 'pcr_runs'}"
if args.smoke:
    E.update(epochs=1, batch_queries=3, hard_negs=1, time_budget_min=3, eval_minutes=99)
OUT = f"{RUNS}/{args.exp}"; os.makedirs(OUT, exist_ok=True)
log = open(f"{OUT}/log.jsonl", "w")
t0 = time.time()
def say(**kw):
    kw["t"] = round(time.time() - t0); log.write(json.dumps(kw) + "\n"); log.flush()
    print(args.exp, json.dumps(kw), flush=True)
seed = E.get("seed", C["seed"]); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------------------------------------------ model (+ teacher)
model = L.build_model(E, C["checkpoints"], M, smoke=args.smoke).to(dev)
lm = model.backbone.lm
n_prompt = L.n_prompt_of(E)
win = L.window_size(lm, n_prompt, E["win_tokens"])
total, trainable = L.count_params(model)
teacher = None
if E.get("teacher"):
    TE = exp_cfg(E["teacher"])
    tpath = f"{RUNS}/{E['teacher']}/best.pt"
    if not os.path.exists(tpath):
        tpath = f"{L.find_runs('best.pt')[E['teacher']]}/best.pt"
    teacher = L.build_model(TE, C["checkpoints"], M, smoke=args.smoke)
    L.load_trained(teacher, torch.load(tpath, map_location="cpu", weights_only=False)["state"])
    teacher.to(dev).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    assert L.window_size(teacher.backbone.lm, L.n_prompt_of(TE), TE["win_tokens"]) == win and L.n_prompt_of(TE) == n_prompt, \
        "teacher and student must read the same windows: same window size and prompt length"

# ------------------------------------------------------------------ data (tokenised once in N8)
train, dv = L.Split("train"), L.Split("dev")
hardneg = np.load(f"{L.CACHE}/train_hardneg.npy")
minipool = json.load(open(f"{L.CACHE}/dev_minipool.json"))
sources = [i for i in range(train.n_q()) if len(train.gold_idx[i])]
if args.smoke:
    sources, minipool = sources[:8], minipool[:3]
assert sources, "no training query has a relevant candidate inside its pool — check train.json / train_candidates.json"
rng = np.random.RandomState(seed)
gold_sets = [set(g.tolist()) for g in train.gold_idx]

def window_of(tok):
    ws = L.doc_windows(len(tok), win)
    a, b = ws[rng.randint(len(ws))]
    return tok[a:b]

def sample_negs(i, n):
    lo, hi = E["hn_range"]
    pool = [int(x) for x in hardneg[i][lo:hi] if x >= 0]
    out = []
    for _ in range(n):
        if pool and rng.rand() < E["hard_frac"]:
            out.append(pool[rng.randint(len(pool))])
        else:
            for _ in range(20):
                j = int(rng.randint(train.n_c()))
                if j not in gold_sets[i]:
                    break
            out.append(j)
    return out

def make_batch(qs):
    """-> query windows, candidate windows (one positive per query first, then hard / random negatives),
    pos_index (the column of each query's positive), ignore (B, M) = candidates that are relevant to a query but are not
    the positive chosen for it (other relevant cases must not count as negatives)"""
    q_seqs = [window_of(train.q_tokens(i)) for i in qs]
    pos = [int(rng.choice(train.gold_idx[i])) for i in qs]
    negs = [j for i in qs for j in sample_negs(i, E["hard_negs"])]
    cands = pos + negs
    c_seqs = [window_of(train.c_tokens(j)) for j in cands]
    ignore = torch.tensor([[(cands[m] in gold_sets[i]) and m != k for m in range(len(cands))] for k, i in enumerate(qs)],
                          dtype=torch.bool, device=dev)
    return q_seqs, c_seqs, torch.arange(len(qs), device=dev), ignore

say(event="start", arch=E["arch"], method=E["method"], params_total=total, params_trainable=trainable,
    trainable_pct=round(100 * trainable / total, 3), window=win, prompt_tokens=n_prompt, token_pool=E["token_pool"],
    train_queries=len(sources), val_queries=len(minipool), time_budget_min=E["time_budget_min"], teacher=E.get("teacher"))

# ------------------------------------------------------------------ validation: re-ranking of BM25 candidates
@torch.no_grad()
def val_f1():
    model.eval()
    cand = sorted({j for m in minipool for j in m})
    pos = {j: k for k, j in enumerate(cand)}
    def first(tok):
        a, b = L.doc_windows(len(tok), win)[0]
        return tok[a:b]
    seqs = [first(dv.q_tokens(i)) for i in range(len(minipool))]
    cseq = [first(dv.c_tokens(j)) for j in cand]
    Z = L.encode(model, seqs + cseq, n_prompt, dev, E["token_pool"])
    q, c = Z[:len(seqs)], Z[len(seqs):]
    top, gold = [], []
    for i, m in enumerate(minipool):
        s = (q[i:i + 1] @ c[[pos[j] for j in m]].T)[0]
        top.append([m[k] for k in s.topk(min(L.K, len(m))).indices.tolist()])
        gold.append(dv.gold_idx[i])
    tp = sum(len(set(t) & set(g.tolist())) for t, g in zip(top, gold))
    n_gold = sum(len(dv.gold[i]) for i in range(len(minipool)))
    n = sum(len(t) for t in top)
    model.train()
    return 2 * tp / max(1, n + n_gold)      # = 2tp / (2tp + fp + fn) with fp = n - tp, fn = n_gold - tp

def save_best(ep, vf):
    torch.save({"state": L.trainable_state(model), "exp": E, "epoch": ep, "val_f1": vf}, f"{OUT}/best.pt")

if E["method"] == "none":                       # zero-shot reference: no training at all
    model.eval()
    vf = val_f1()
    save_best(0.0, vf)
    summary = {"exp": args.exp, "arch": E["arch"], "method": "none", "teacher": None, "student_layers": E.get("student_layers"),
               "window": win, "prompt_tokens": 0, "token_pool": E["token_pool"], "seed": seed, "best_epoch": 0.0,
               "epochs_done": 0.0, "steps": 0, "params_total": total, "params_trainable": 0, "peak_gpu_gb": None,
               "val_f1": round(vf, 4), "train_minutes": round((time.time() - t0) / 60, 1)}
    json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
    say(event="done", **summary)
    raise SystemExit(0)

# ------------------------------------------------------------------ training setup
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=E["lr"], weight_decay=E["weight_decay"])
B = E["batch_queries"]
steps_per_epoch = max(1, int(np.ceil(len(sources) / B)))
budget_s = E["time_budget_min"] * 60
total_steps = max(1, int(steps_per_epoch * E["epochs"]))
warm = max(1, int(E["warmup_frac"] * min(total_steps, steps_per_epoch)))
def lr_at(step):            # warm-up, then linear decay to 10% over the planned steps or the time budget, whichever ends first
    frac = max(step / total_steps, (time.time() - t0) / budget_s)
    return E["lr"] * ((step + 1) / warm if step < warm else max(0.1, 1.0 - frac))
scaler = torch.amp.GradScaler("cuda", enabled=cuda)

# ------------------------------------------------------------------ train
best, best_epoch, step, last_eval, stop = -1.0, 0.0, 0, time.time(), False
model.train()
if cuda:
    torch.cuda.reset_peak_memory_stats()
epoch = 0
while not stop:
    epoch += 1
    losses, nces, kds = [], [], []
    order = rng.permutation(len(sources))
    for bi in range(0, len(order), B):
        if (args.smoke and bi // B >= 3) or step >= total_steps or time.time() - t0 > budget_s:
            stop = True
            break
        qs = [sources[j] for j in order[bi:bi + B]]
        q_seqs, c_seqs, pos_index, ignore = make_batch(qs)
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        Z = L.encode(model, q_seqs + c_seqs, n_prompt, dev, E["token_pool"], batch_tokens=10 ** 9)
        q, c = Z[:len(q_seqs)], Z[len(q_seqs):]
        nce = M.info_nce_loss(q, c, pos_index, E["temperature"], ignore)
        loss = nce
        if teacher is not None:
            with torch.no_grad():
                Zt = L.encode(teacher, q_seqs + c_seqs, n_prompt, dev, TE["token_pool"], batch_tokens=10 ** 9)
                t_scores = Zt[:len(q_seqs)] @ Zt[len(q_seqs):].T
            kd = M.kd_scores_loss(q @ c.T, t_scores, E["kd"]["temperature"], ignore)
            loss = E["kd"]["alpha"] * nce + (1.0 - E["kd"]["alpha"]) * kd
            kds.append(float(kd))
        aux = getattr(lm, "aux_loss", None)
        if E["moe_aux_coef"] and aux is not None:
            loss = loss + E["moe_aux_coef"] * aux
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, E["grad_clip"])
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        losses.append(loss.item()); nces.append(float(nce)); step += 1
        if time.time() - last_eval > E["eval_minutes"] * 60:
            vf = val_f1(); last_eval = time.time()
            ep = round(step / steps_per_epoch, 3)
            say(event="eval", epoch=ep, step=step, train_loss=round(float(np.mean(losses[-100:])), 4),
                nce=round(float(np.mean(nces[-100:])), 4), val_f1=round(vf, 4))
            if vf > best:
                best, best_epoch = vf, ep
                save_best(ep, vf)
    if step >= total_steps:
        stop = True
    ep = round(step / steps_per_epoch, 3)
    if losses:
        say(event="epoch_end" if not stop else "train_end", epoch=ep, step=step, train_loss=round(float(np.mean(losses)), 4),
            nce=round(float(np.mean(nces)), 4), kd=round(float(np.mean(kds)), 4) if kds else None)
    vf = val_f1(); last_eval = time.time()
    say(event="eval", epoch=ep, step=step, val_f1=round(vf, 4))
    if vf > best:
        best, best_epoch = vf, ep
        save_best(ep, vf)

peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else None
summary = {"exp": args.exp, "arch": E["arch"], "method": E["method"], "teacher": E.get("teacher"),
           "student_layers": E.get("student_layers"), "window": win, "prompt_tokens": n_prompt,
           "token_pool": E["token_pool"], "seed": seed, "best_epoch": best_epoch,
           "epochs_done": round(step / steps_per_epoch, 3), "steps": step, "params_total": total,
           "params_trainable": trainable, "peak_gpu_gb": peak, "val_f1": round(best, 4),
           "train_minutes": round((time.time() - t0) / 60, 1)}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
say(event="done", **summary)

In [ ]:
%%writefile embed_pcr.py
"""Embeds the DEV and TEST queries and candidate pools with ONE finished run (its best checkpoint): every document becomes
up to embed_windows windows (evenly spread over the document, first and last included), every window one L2-normalised
vector. The lab turns these into query-candidate scores (several ways to combine windows) and zips. Writes
pcr_emb/<run>/{dev,test}_{q,c}_{emb,off,widx}.npy + meta.json; a finished run is skipped (resumable).
   python embed_pcr.py --run NAME --device cuda:0   [--smoke]"""
import argparse, json, os, time
import numpy as np, torch
import pcr_common as L
import pcr_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--run", required=True); ap.add_argument("--device", default="cuda:0")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
dev = args.device
if dev.startswith("cuda"):
    torch.cuda.set_device(dev)
C = json.load(open(f"{L.WORK}/run_config_pcr.json"))
folder = L.find_runs("best.pt", "pcr_runs_smoke" if args.smoke else "pcr_runs")[args.run]
ck = torch.load(f"{folder}/best.pt", map_location="cpu", weights_only=False)
E = ck["exp"]
summary = json.load(open(f"{folder}/summary.json")) if os.path.exists(f"{folder}/summary.json") else {}
OUT = f"{L.WORK}/{'pcr_emb_smoke' if args.smoke else 'pcr_emb'}/{args.run}"
if os.path.exists(f"{OUT}/meta.json"):
    print(f"{args.run}: embeddings exist, skipped"); raise SystemExit(0)
os.makedirs(OUT, exist_ok=True)

model = L.build_model(E, C["checkpoints"], M, smoke=args.smoke)
L.load_trained(model, ck["state"])
L.merge_adapters(model)
model.to(dev).eval()
n_prompt = L.n_prompt_of(E)
win = L.window_size(model.backbone.lm, n_prompt, E["win_tokens"])
cap = C["lab"]["embed_windows"]
SMOKE_Q, SMOKE_C = 8, 60
t0 = time.time()
print(f"{args.run}: {E['arch']} {E['method']} | window {win} tokens, up to {cap} windows per document, "
      f"token pool {E['token_pool']}", flush=True)

@torch.no_grad()
def embed_docs(get, n):
    seqs, off, widx = [], [0], []
    for i in range(n):
        t = get(i)
        ws = L.doc_windows(len(t), win)
        for k in L.pick_windows(ws, cap):
            a, b = ws[k]
            seqs.append(t[a:b]); widx.append(k)
        off.append(len(seqs))
    out = []
    for a in range(0, len(seqs), 8192):
        out.append(L.encode(model, seqs[a:a + 8192], n_prompt, dev, E["token_pool"]).half().cpu().numpy())
        print(f"    {min(a + 8192, len(seqs))}/{len(seqs)} windows, {time.time() - t0:.0f}s", flush=True)
    return np.concatenate(out), np.asarray(off, dtype=np.int64), np.asarray(widx, dtype=np.int16)

info = {}
for split in ("dev", "test"):
    S = L.Split(split)
    nq, nc = (min(S.n_q(), SMOKE_Q), min(S.n_c(), SMOKE_C)) if args.smoke else (S.n_q(), S.n_c())
    for kind, n, get in (("q", nq, S.q_tokens), ("c", nc, S.c_tokens)):
        print(f"  {split} {kind}: {n} documents", flush=True)
        emb, off, widx = embed_docs(get, n)
        np.save(f"{OUT}/{split}_{kind}_emb.npy", emb); np.save(f"{OUT}/{split}_{kind}_off.npy", off)
        np.save(f"{OUT}/{split}_{kind}_widx.npy", widx)
        info[f"{split}_{kind}"] = {"docs": n, "windows": int(len(emb))}
sec = time.time() - t0
json.dump({"run": args.run, "arch": E["arch"], "method": E["method"], "teacher": E.get("teacher"), "window": win,
           "embed_windows": cap, "token_pool": E["token_pool"], "seconds": round(sec, 1), "info": info,
           "params_total": summary.get("params_total"), "params_trainable": summary.get("params_trainable"),
           "val_f1": summary.get("val_f1")}, open(f"{OUT}/meta.json", "w"))
print(f"{args.run}: done in {sec:.0f}s", flush=True)

In [ ]:
%%writefile pcr_lab.py
"""Phase 7 lab — no training. Reads the window embeddings of every finished run (pcr_emb/<run>/), turns them into
query-candidate scores in several ways, scores each on the DEV queries with the official metric (Micro-F1@7 of
evaluate_phase7.py), and writes a Codabench zip (test predictions: exactly 7 candidates from the test pool per query)
for the best variants, submissions/index.csv and a score chart.

Variants per run:  window aggregation (first window / cosine of mean embeddings / MaxSim over windows, one- and two-sided /
best window pair)  x  hubness correction (CSLS: subtract beta x a candidate's mean similarity to its nearest queries of
the same split; uses no labels)  ->  then the best of each run is fused with BM25 (z-scored sum, several weights), and
the runs are ensembled (z-score average of the k best runs, k = 2..n), again with and without BM25.
BM25-only (over the Phase-1 tokens) is a reference row and is never zipped: it is not a Phase-1 model.
Tentative score = dev Micro-F1@7. The first val_queries dev queries also picked each run's checkpoint, so dev_f1_rest
(all other dev queries) is the cleaner number. You choose which zips to upload."""
import csv, glob, hashlib, json, os
import numpy as np, torch
import pcr_common as L

ARCH_COLORS = {"decoder": "#2a78d6", "prefix": "#eb6834", "encdec": "#1baf7a", "swa": "#eda100", "moe": "#e87ba4",
               "ensemble": "#7b61c4"}
METRIC_GREYS = ("#3d3c3a", "#8f8d88")

def z(S):
    S = np.asarray(S, dtype=np.float32)
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)

class Lab:
    def __init__(self, out_dir=f"{L.WORK}/submissions", sub="pcr_emb"):
        C = json.load(open(f"{L.WORK}/run_config_pcr.json"))
        self.cfg, self.val_q, self.sub = C["lab"], C["val_queries"], sub
        self.dv, self.te = L.Split("dev"), L.Split("test")
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.dtype = torch.float16 if self.dev == "cuda" else torch.float32
        bm = C["bm25"]
        self.bm = {"dev": L.bm25_matrix(self.dv, **bm), "test": L.bm25_matrix(self.te, **bm)}
        self.out_dir = out_dir
        os.makedirs(f"{out_dir}/dev_predictions", exist_ok=True)
        self.rows, self.bases = [], []
        print(f"dev: {self.dv.n_q()} queries x {self.dv.n_c()} candidates | test: {self.te.n_q()} x {self.te.n_c()}")

    # ------------------------------------------------------------ scoring one score matrix
    def top7(self, S, split):
        S = torch.as_tensor(np.asarray(S, dtype=np.float32)).clone()
        split.mask_self(S)
        assert S.shape[1] >= L.K, "candidate pool smaller than 7"
        return torch.topk(S, L.K, dim=1).indices.numpy()

    def add(self, name, Sd, St, zippable=True, **meta):
        dtop, ttop = self.top7(Sd, self.dv), self.top7(St, self.te)
        f1, p, r = L.micro_f1(dtop, self.dv.gold_idx, self.dv.n_gold)
        V = self.val_q
        f1r = L.micro_f1(dtop[V:], self.dv.gold_idx[V:], sum(len(g) for g in self.dv.gold[V:]))[0] if self.dv.n_q() > V else None
        row = {"name": name, "dev_f1": round(f1, 4), "dev_f1_rest": None if f1r is None else round(f1r, 4),
               "dev_precision": round(p, 4), "dev_recall": round(r, 4), "run": "-", "arch": "-", "method": "-", "kind": "-",
               "agg": "", "csls": "", "bm25_w": "", "members": "", "val_f1": None, "trainable_pct": None, "params_total": None,
               "_dtop": dtop, "_ttop": ttop, "_zip": zippable}
        row.update(meta)
        self.rows.append(row)
        return row

    # ------------------------------------------------------------ window embeddings -> document scores
    def collect(self):
        found = {}
        for rank, root in enumerate((L.WORK, L.INPUT_ROOT)):
            for p in glob.glob(f"{root}/**/{self.sub}/*/meta.json", recursive=True):
                d = os.path.dirname(p); run = os.path.basename(d)
                key = (rank, -os.path.getmtime(p))
                if run not in found or key < found[run][0]:
                    found[run] = (key, d)
        out = {r: d for r, (_, d) in sorted(found.items())}
        print(f"embedded runs found: {len(out)}: {list(out)}")
        return out

    def load_padded(self, d, split, kind, n_expected):
        emb = np.load(f"{d}/{split}_{kind}_emb.npy")
        off = np.load(f"{d}/{split}_{kind}_off.npy")
        n = len(off) - 1
        if n != n_expected:
            raise ValueError(f"{d}: {split} {kind} has {n} documents, the data has {n_expected} (a smoke-test embedding?)")
        w = np.diff(off)
        W = int(w.max())
        pad = np.zeros((n, W, emb.shape[1]), dtype=np.float16)
        doc = np.repeat(np.arange(n), w)
        pad[doc, np.arange(len(emb)) - off[:-1][doc]] = emb
        mask = np.arange(W)[None, :] < w[:, None]
        return torch.from_numpy(pad).to(self.dev, self.dtype), torch.from_numpy(mask).to(self.dev)

    @torch.no_grad()
    def doc_scores(self, Q, Qm, C, Cm, how):
        if how == "first":
            return (Q[:, 0].float() @ C[:, 0].float().T).cpu().numpy()
        if how == "mean":
            qd = torch.nn.functional.normalize((Q.float() * Qm[..., None]).sum(1), dim=-1)
            cd = torch.nn.functional.normalize((C.float() * Cm[..., None]).sum(1), dim=-1)
            return (qd @ cd.T).cpu().numpy()
        Nq, Wq, _ = Q.shape
        Nc, Wc, _ = C.shape
        qb = max(1, int(1.2e8 // (Nc * Wq * Wc)))
        out = []
        for a in range(0, Nq, qb):
            q, qm = Q[a:a + qb], Qm[a:a + qb]
            sims = torch.einsum("qad,cbd->qcab", q, C).masked_fill(~Cm[None, :, None, :], float("-inf"))
            mq = sims.max(3).values                                            # (qb, Nc, Wq): best candidate window per query window
            if how == "maxall":
                s = mq.masked_fill(~qm[:, None, :], float("-inf")).max(2).values.float()
            else:
                sq = (mq.float() * qm[:, None, :]).sum(2) / qm.sum(1)[:, None]    # mean over the query's windows
                if how == "maxsim_q":
                    s = sq
                else:                                                          # maxsim_sym: + the other direction
                    mc = sims.masked_fill(~qm[:, None, :, None], float("-inf")).max(2).values.masked_fill(~Cm[None], 0)
                    sc = (mc.float() * Cm[None]).sum(2) / Cm.sum(1)[None]
                    s = (sq + sc) / 2
            out.append(s.cpu().numpy())
            del sims, mq
        return np.concatenate(out)

    def csls(self, S, beta):
        if not beta:
            return S
        r = torch.from_numpy(S).topk(min(self.cfg["csls_k"], S.shape[0]), dim=0).values.mean(0).numpy()
        return S - beta * r[None, :]

    # ------------------------------------------------------------ variants
    def run_dense(self, run, d):
        meta = json.load(open(f"{d}/meta.json"))
        tp = meta.get("params_trainable"), meta.get("params_total")
        common = dict(run=run, arch=meta["arch"], method=meta["method"] + (f" (KD from {meta['teacher']})" if meta.get("teacher") else ""),
                      kind="dense", val_f1=meta.get("val_f1"), params_total=tp[1],
                      trainable_pct=round(100 * tp[0] / tp[1], 3) if tp[0] is not None and tp[1] else None)
        Qd, Qdm = self.load_padded(d, "dev", "q", self.dv.n_q()); Cd, Cdm = self.load_padded(d, "dev", "c", self.dv.n_c())
        Qt, Qtm = self.load_padded(d, "test", "q", self.te.n_q()); Ct, Ctm = self.load_padded(d, "test", "c", self.te.n_c())
        best = None
        for how in self.cfg["aggs"]:
            Sd, St = self.doc_scores(Qd, Qdm, Cd, Cdm, how), self.doc_scores(Qt, Qtm, Ct, Ctm, how)
            line = []
            for beta in self.cfg["csls"]:
                Sd2, St2 = self.csls(Sd, beta), self.csls(St, beta)
                row = self.add(f"{run}__{how}__csls{beta:g}", Sd2, St2, agg=how, csls=beta, **common)
                line.append(f"csls {beta:g}: {row['dev_f1']:.4f}")
                if best is None or row["dev_f1"] > best[0]["dev_f1"]:
                    best = (row, Sd2, St2)
            print(f"  {run:24s} {how:11s} " + " | ".join(line), flush=True)
        del Qd, Cd, Qt, Ct
        if self.dev == "cuda":
            torch.cuda.empty_cache()
        self.bases.append(best)

    def fusion_rows(self, base_name, Sd, St, kind, **meta):
        for w in self.cfg["fusion_w"]:
            if w > 0:
                self.add(f"{base_name}__bm25w{w:g}", z(Sd) + w * z(self.bm["dev"]), z(St) + w * z(self.bm["test"]),
                         kind=kind + "+bm25", bm25_w=w, **meta)

    def finish_variants(self):
        for row, Sd, St in list(self.bases):
            m = {k: row[k] for k in ("run", "arch", "method", "agg", "csls", "val_f1", "trainable_pct", "params_total")}
            self.fusion_rows(row["name"], Sd, St, "dense", **m)
        order = sorted(self.bases, key=lambda b: -b[0]["dev_f1"])
        if self.cfg.get("ensembles", True):
            for k in range(2, len(order) + 1):
                mem = order[:k]
                Sd = np.mean([z(b[1]) for b in mem], 0); St = np.mean([z(b[2]) for b in mem], 0)
                name = f"ens{k}"
                members = "+".join(b[0]["run"] for b in mem)
                meta = dict(run="ensemble", arch="ensemble", method=f"mean of z-scores of the {k} best runs", members=members)
                self.add(name, Sd, St, kind="ensemble", **meta)
                self.fusion_rows(name, Sd, St, "ensemble", **meta)
        self.add("(reference) BM25 only over Phase-1 tokens", self.bm["dev"], self.bm["test"], zippable=False,
                 kind="reference", method="lexical (not a Phase-1 model)")
        ref = self.rows[-1]
        print(f"\nreference BM25 only: dev F1@7 {ref['dev_f1']:.4f}")

    # ------------------------------------------------------------ zips + index
    def finish(self, top_n=40):
        key = lambda r: (-r["dev_f1"], -(r["dev_f1_rest"] or 0))
        order = sorted(self.rows, key=key)
        zippable = [r for r in order if r["_zip"]]
        keep = {id(r) for r in zippable[:top_n]}
        best_of = {}
        for r in zippable:
            best_of.setdefault(r["run"], r)
        keep |= {id(r) for r in best_of.values()}
        for f in glob.glob(f"{self.out_dir}/*.zip") + glob.glob(f"{self.out_dir}/dev_predictions/*__*.json"):
            os.remove(f)
        json.dump({q: g for q, g in zip(self.dv.q_ids, self.dv.gold)}, open(f"{self.out_dir}/dev_predictions/dev_truth.json", "w"))
        json.dump(self.dv.c_ids, open(f"{self.out_dir}/dev_predictions/dev_candidate_ids.json", "w"))
        seen, n = {}, 0
        for r in order:
            r["file"] = ""
            if id(r) not in keep:
                continue
            h = hashlib.md5(json.dumps(r["_ttop"].tolist()).encode()).hexdigest()
            if h in seen:
                r["file"] = f"(same as {seen[h]})"
                continue
            seen[h] = r["file"] = f"{r['name']}.zip"
            L.write_submission(L.prediction_dict(self.te.q_ids, self.te.c_ids, r["_ttop"]), f"{self.out_dir}/_prediction.json",
                               f"{self.out_dir}/{r['file']}")
            L.write_submission(L.prediction_dict(self.dv.q_ids, self.dv.c_ids, r["_dtop"]),
                               f"{self.out_dir}/dev_predictions/{r['name']}.json")
            n += 1
        if os.path.exists(f"{self.out_dir}/_prediction.json"):
            os.remove(f"{self.out_dir}/_prediction.json")
        cols = ["name", "dev_f1", "dev_f1_rest", "dev_precision", "dev_recall", "kind", "run", "arch", "method", "agg", "csls",
                "bm25_w", "members", "val_f1", "trainable_pct", "params_total", "file"]
        with open(f"{self.out_dir}/index.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
            w.writeheader()
            for r in order:
                w.writerow(r)
        print(f"{len(self.rows)} variants scored, {n} zips written to {self.out_dir}/ (index.csv lists all)")
        return order

def table(rows, n=40):
    print(f"{'dev F1':>6s} {'rest':>6s} {'P@7':>6s} {'R@7':>6s}  {'variant':72s} zip")
    for r in rows[:n]:
        rest = "   n/a" if r["dev_f1_rest"] is None else f"{r['dev_f1_rest']:.4f}"
        print(f"{r['dev_f1']:.4f} {rest:>6s} {r['dev_precision']:.4f} {r['dev_recall']:.4f}  {r['name'][:72]:72s} {r.get('file', '')}")

def chart(rows, path, n=25):
    """left: the n best zippable variants by dev Micro-F1@7, coloured by Phase-1 architecture (ensembles purple), the BM25-only
    reference as a dashed line; right: per run the best dense-only variant and the best variant fused with BM25.
    index.csv is the table view of the same numbers."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ref = next((r for r in rows if r["kind"] == "reference"), None)
    cand = [r for r in rows if r["kind"] != "reference"]
    top = cand[:n][::-1]
    groups = {}
    for r in cand:
        g = groups.setdefault(r["run"], {})
        g.setdefault("fused" if r["kind"].endswith("+bm25") else "dense", r)
    glist = sorted(groups.items(), key=lambda kv: -max(v["dev_f1"] for v in kv[1].values()))[:14][::-1]
    ink, muted, grid, bg = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(17, max(6, 0.32 * max(len(top), 2.6 * len(glist)))), gridspec_kw={"width_ratios": [1.15, 1]})
    fig.patch.set_facecolor(bg)
    for ax in (a1, a2):
        ax.set_facecolor(bg)
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.spines["bottom"].set_color(grid)
        ax.tick_params(colors=muted, labelsize=8, length=0)
        ax.grid(axis="x", color=grid, linewidth=0.8)
        ax.set_axisbelow(True)
    y = np.arange(len(top))
    a1.barh(y, [r["dev_f1"] for r in top], height=0.62, color=[ARCH_COLORS.get(r["arch"], muted) for r in top])
    a1.set_yticks(y, [r["name"][:62] for r in top], fontsize=7.5, color=ink)
    for yi, r in zip(y, top):
        a1.text(r["dev_f1"] + 0.002, yi, f"{r['dev_f1']:.3f}", va="center", fontsize=7.5, color=muted)
    hi = max(r["dev_f1"] for r in top)
    lo = min([r["dev_f1"] for r in top] + ([ref["dev_f1"]] if ref else []))
    a1.set_xlim(max(0, lo - 0.04), hi * 1.06)
    if ref:
        for ax in (a1, a2):
            ax.axvline(ref["dev_f1"], color=muted, linestyle="--", linewidth=1)
        a1.text(ref["dev_f1"], len(top) - 0.3, " BM25 only", color=muted, fontsize=8, va="bottom")
    a1.set_xlabel("dev Micro-F1@7 (tentative)", color=muted, fontsize=9)
    a1.set_title(f"Top {len(top)} variants (run__window-aggregation__csls__bm25 weight)", loc="left", color=ink, fontsize=11)
    archs = [a for a in ARCH_COLORS if any(r["arch"] == a for r in top)]
    a1.legend([plt.Rectangle((0, 0), 1, 1, color=ARCH_COLORS[a]) for a in archs], archs, loc="lower right", frameon=False, fontsize=8)
    yy = np.arange(len(glist)); h = 0.38
    for k, (key, lab) in enumerate((("dense", "dense only (best variant)"), ("fused", "fused with BM25 (best variant)"))):
        vals = [g.get(key, {}).get("dev_f1", 0) for _, g in glist]
        a2.barh(yy + (0.5 - k) * h, vals, height=h, color=METRIC_GREYS[k], label=lab, edgecolor=bg, linewidth=1.5)
        for yi, v in zip(yy, vals):
            if v:
                a2.text(v + 0.002, yi + (0.5 - k) * h, f"{v:.3f}", va="center", fontsize=7, color=muted)
    a2.set_yticks(yy, [name for name, _ in glist], fontsize=8, color=ink)
    a2.set_xlim(max(0, lo - 0.04), hi * 1.06)
    a2.set_xlabel("dev Micro-F1@7 per run", color=muted, fontsize=9)
    a2.set_title("Best per run", loc="left", color=ink, fontsize=11)
    a2.legend(loc="lower right", frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=130, facecolor=fig.get_facecolor())
    plt.close(fig)
    print("chart saved:", path)

In [ ]:
%%writefile evaluate_phase7.py
#!/usr/bin/env python3
"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).

It uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /
Micro-Recall@7 definitions as the Codabench scorer.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

K = 7


def reject_constant(value: str) -> None:
    raise ValueError(f"Non-standard JSON numeric value '{value}' is not allowed.")


def pairs_no_duplicates(pairs):
    result = {}
    duplicates = []
    for key, value in pairs:
        if key in result:
            duplicates.append(key)
        result[key] = value
    if duplicates:
        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")
    return result


def load_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"
        ) from exc


def load_truth(path: Path) -> dict[str, list[str]]:
    data = load_json(path)
    # Accept either the simple gold.json mapping or a Phase-7 dev query file:
    # {query_id: {text: [...], relevant_candidates: [...]}}
    if not isinstance(data, dict) or not data:
        raise ValueError("Truth must be a non-empty JSON object.")
    out: dict[str, list[str]] = {}
    for query_id, value in data.items():
        if isinstance(value, dict) and "relevant_candidates" in value:
            labels = value["relevant_candidates"]
        else:
            labels = value
        if not isinstance(query_id, str) or not query_id.strip():
            raise ValueError("Truth contains an empty/invalid query ID.")
        if not isinstance(labels, list):
            raise ValueError(f"Truth for query '{query_id}' must contain a list of relevant candidate IDs.")
        cleaned=[]; seen=set()
        for idx,candidate_id in enumerate(labels):
            if not isinstance(candidate_id,str) or not candidate_id.strip():
                # Treat blank source labels as absent labels, matching the builder.
                continue
            if candidate_id in seen:
                raise ValueError(f"Truth for query '{query_id}' contains duplicate candidate ID '{candidate_id}'.")
            seen.add(candidate_id); cleaned.append(candidate_id)
        out[query_id]=cleaned
    return out


def load_candidate_ids(path: Path) -> set[str]:
    data = load_json(path)
    if not isinstance(data, list) or not data:
        raise ValueError("candidate_ids must be a non-empty JSON list.")
    ids=[]; seen=set()
    for idx,v in enumerate(data):
        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")
        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID '{v}'.")
        seen.add(v); ids.append(v)
    return set(ids)


def load_prediction(path: Path) -> Any:
    return load_json(path)


def validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:
    errors=[]
    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]
    gold_ids=set(gold); pred_ids=set(pred)
    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID '{q}'.")
    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID '{q}'.")
    for q in sorted(gold_ids & pred_ids):
        vals=pred[q]
        if not isinstance(vals,list):
            errors.append(f"Query '{q}': prediction must be a list."); continue
        if len(vals)!=K: errors.append(f"Query '{q}': expected exactly {K} candidates, got {len(vals)}.")
        seen=set()
        for i,c in enumerate(vals):
            if not isinstance(c,str) or not c.strip(): errors.append(f"Query '{q}': index {i} is not a non-empty string."); continue
            if c in seen: errors.append(f"Query '{q}': duplicate candidate ID '{c}'.")
            seen.add(c)
            if c not in candidates: errors.append(f"Query '{q}': candidate ID '{c}' is outside the candidate pool.")
    return errors


def score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:
    tp=fp=fn=0
    for q,labels in gold.items():
        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}


def parse_args():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--truth",type=Path,required=True,help="gold.json or Phase-7 dev.json with relevant_candidates")
    p.add_argument("--prediction",type=Path,required=True)
    p.add_argument("--candidate-ids",type=Path,required=True,help="candidate_ids.json")
    p.add_argument("--output",type=Path,default=None,help="Optional output scores JSON path")
    return p.parse_args()


def main():
    a=parse_args()
    gold=load_truth(a.truth); candidates=load_candidate_ids(a.candidate_ids); pred=load_prediction(a.prediction)
    errs=validate(pred,gold,candidates)
    if errs:
        print("LOCAL VALIDATION FAILED")
        for i,e in enumerate(errs,1): print(f"  {i}. {e}")
        print(f"Total validation errors: {len(errs)}")
        raise SystemExit(1)
    result=score(gold,pred)
    scores={k:result[k] for k in ("primary_metric_1","secondary_metric_1","secondary_metric_2")}
    print("Validation passed.")
    print(f"Micro-F1@7: {scores['primary_metric_1']:.10f}")
    print(f"Micro-Precision@7: {scores['secondary_metric_1']:.10f}")
    print(f"Micro-Recall@7: {scores['secondary_metric_2']:.10f}")
    print(f"TP={result['tp']} FP={result['fp']} FN={result['fn']}")
    if a.output:
        a.output.parent.mkdir(parents=True,exist_ok=True)
        with a.output.open('w',encoding='utf-8') as f: json.dump(scores,f,indent=2); f.write('\n')
        print(f"Wrote {a.output}")

if __name__=='__main__': main()

## N7b. Pre-flight check — every input present, your two losses written? (stops here if not)

In [ ]:
# N7b — PRE-FLIGHT: checks every input this session needs and STOPS here (within a minute, not after hours) if one
# is missing. Read the printout: every line must show a path, not MISSING.
import json, glob, os, importlib, torch
import torch.nn.functional as F
import pcr_common as L
C = json.load(open("/kaggle/working/run_config_pcr.json"))
problems = []
print("tokenizer:", L.find_input("bpe_tok.py"), "|", L.find_input("bpe.json"))
for f in ("train.json", "train_candidates.json", "dev.json", "dev_candidates.json", "test.json", "test_candidates.json"):
    p = f"{L.DATA}/{f}"
    print(f"data {f}: {'%.1f MB' % (os.path.getsize(p) / 1e6) if os.path.exists(p) else 'MISSING'}")
    if not os.path.exists(p):
        problems.append(f"{f} was not downloaded (N1)")
archs = set()
for e in SESSION_EXPERIMENTS + [r for r in EMBED_RUNS if r in C["experiments"]]:
    x = C["experiments"][e]
    archs.add(x["arch"])
    if x.get("teacher"):
        archs.add(C["experiments"][x["teacher"]]["arch"])
for arch in sorted(archs):
    pat = C["checkpoints"][arch]
    hits = sorted(glob.glob(f"{L.INPUT_ROOT}/**/{pat}", recursive=True))
    print(f"Phase-1 checkpoint {arch:8s} ({pat}): {hits[0] if hits else 'MISSING'}")
    if not hits:
        problems.append(f"the Phase-1 '{arch}' checkpoint '{pat}' is not in any input — add the notebook / dataset "
                        f"that contains it" + (" (decoder: notebook27acca3fc3 = notebook D)" if arch == "decoder" else
                                               " (moe: Phase1_training2 / phase1-train-extra)" if arch == "moe" else ""))
for e in SESSION_EXPERIMENTS:
    t = C["experiments"][e].get("teacher")
    if t and t not in SESSION_EXPERIMENTS:
        f = L.find_runs("best.pt").get(t)
        print(f"distillation teacher '{t}' for {e}: {f + '/best.pt' if f else 'MISSING'}")
        if not f:
            problems.append(f"the trained teacher '{t}' was not found — add it to SESSION_EXPERIMENTS")
for r in EMBED_RUNS:
    if r not in SESSION_EXPERIMENTS and r not in L.find_runs("best.pt"):
        problems.append(f"EMBED_RUNS: '{r}' is neither trained this session nor found in the inputs")
import pcr_methods as M
importlib.reload(M)
# --- your two losses on small random tensors: they must run, be finite, have a gradient, and honour the ignore mask
torch.manual_seed(0)
try:
    q = F.normalize(torch.randn(3, 8), dim=-1).requires_grad_(True)
    c = F.normalize(torch.randn(9, 8), dim=-1)
    c[5] = q.detach()[0]                         # a candidate identical to query 0, not its positive
    pos = torch.arange(3)
    ign = torch.zeros(3, 9, dtype=torch.bool); ign[0, 5] = True
    l_ign, l_all = M.info_nce_loss(q, c, pos, 0.05, ign), M.info_nce_loss(q, c, pos, 0.05, None)
    assert l_ign.dim() == 0 and torch.isfinite(l_ign), "info_nce_loss must return one finite number"
    l_ign.backward()
    assert q.grad is not None and torch.isfinite(q.grad).all(), "info_nce_loss: no gradient reaches the query vectors"
    assert l_ign.item() < l_all.item() - 0.5, "info_nce_loss ignores the `ignore` mask (a relevant case counted as a negative)"
    good = c.clone(); good[:3] = q.detach()
    assert M.info_nce_loss(q.detach(), good, pos, 0.05, None) < M.info_nce_loss(q.detach(), c, pos, 0.05, None), \
        "info_nce_loss does not reward a candidate that is close to its query"
    print("info_nce_loss: OK")
except NotImplementedError:
    problems.append("info_nce_loss in N6 (section 9) is not written yet")
except AssertionError as e:
    problems.append(f"info_nce_loss: {e}")
if any(C["experiments"][e].get("teacher") for e in SESSION_EXPERIMENTS):
    try:
        s, t = torch.randn(3, 9), torch.randn(3, 9)
        s2 = s.clone().requires_grad_(True)
        ign = torch.zeros(3, 9, dtype=torch.bool); ign[1, 4] = True
        k = M.kd_scores_loss(s2, t, 0.05, ign)
        assert k.dim() == 0 and torch.isfinite(k), "kd_scores_loss must return one finite number"
        k.backward()
        assert s2.grad is not None, "kd_scores_loss: no gradient reaches the student scores"
        assert abs(M.kd_scores_loss(t, t, 0.05, None).item()) < 1e-4, "kd_scores_loss of identical distributions must be ~0"
        assert M.kd_scores_loss(s, t, 0.05, None).item() > 0.01, "kd_scores_loss of different distributions must be > 0"
        print("kd_scores_loss: OK")
    except NotImplementedError:
        problems.append("kd_scores_loss in N6 (section 9) is not written yet — write it, or take the distillation runs out of SESSION_EXPERIMENTS")
    except AssertionError as e:
        problems.append(f"kd_scores_loss: {e}")
if any(C["experiments"][e]["method"] == "prompt" for e in SESSION_EXPERIMENTS):
    try:
        m = L.RetModel(L.Backbone("decoder", L.build_lm("decoder", L.TINY["decoder"]))).eval()
        for p in m.parameters():
            p.requires_grad_(False)
        x = torch.tensor([[L.PAD] * 4 + [10, 11, 12, 13, 14]])
        before = m.backbone(x).detach()
        M.METHODS["prompt"](m, n_tokens=4)
        assert [n for n, p in m.named_parameters() if p.requires_grad], "method_prompt created no trainable parameter"
        assert not torch.allclose(before, m.backbone(x).detach()), "the soft prompt does not change the model's output (hook?)"
        print("method_prompt: OK")
    except NotImplementedError:
        problems.append("method_prompt is not written (Phase 6 section 8)")
    except AssertionError as e:
        problems.append(f"method_prompt: {e}")
try:
    import scipy
    print("scipy: OK")
except ImportError as e:
    problems.append(f"scipy missing ({e})")
print("finished runs found in the inputs (they join embedding / the lab):", sorted(L.find_runs("best.pt")) or "none")
if problems:
    raise RuntimeError("PRE-FLIGHT FAILED — nothing was trained:\n  - " + "\n  - ".join(problems))
print("\nPRE-FLIGHT OK")

## N8. Data check + one-time tokenisation + BM25 helper files (sizes, lengths, gold sets, the lexical floor)

In [ ]:
# N8 — read the data, tokenise everything ONCE (all CPU cores; cached in /kaggle/working/p7cache), build the BM25 hard
# negatives, then show sizes, lengths and the BM25-only dev score (the floor your models have to beat or complement).
# If this cell fails, paste the error.
import json, time, importlib, glob, os, shutil, numpy as np
import pcr_common as L
importlib.reload(L)
C = json.load(open("/kaggle/working/run_config_pcr.json"))
prev = sorted(glob.glob(f"{L.INPUT_ROOT}/**/p7cache/test_meta.json", recursive=True))
if prev and not os.path.exists(L.CACHE):                 # an earlier run's cache: no need to tokenise again
    shutil.copytree(os.path.dirname(prev[0]), L.CACHE)
    print("reusing the tokenisation cache from", os.path.dirname(prev[0]))
t0 = time.time()
L.build_cache()
print(f"tokenised in {time.time() - t0:.0f}s")
t0 = time.time()
L.build_aux(C["val_queries"], C["hn_pool"], C["minipool_k"])
print(f"BM25 helper files in {time.time() - t0:.0f}s")
win = C["defaults"]["win_tokens"]
for name in L.SPLITS:
    s = L.Split(name)
    ql, cl = s.q_lengths(), s.c_lengths()
    nw = np.array([len(L.doc_windows(int(n), win)) for n in cl])
    print(f"{name:5s}: {s.n_q():5d} queries, pool {s.n_c():6d} | query tokens median {int(np.median(ql))}, 90% {int(np.percentile(ql, 90))}, max {ql.max()}"
          f" | candidate tokens median {int(np.median(cl))}, 90% {int(np.percentile(cl, 90))}, max {cl.max()}"
          f" | windows of {win}: candidates median {int(np.median(nw))}, {100 * np.mean(nw > C['lab']['embed_windows']):.0f}% above embed_windows")
    if name != "test":
        g = np.array([len(x) for x in s.gold])
        inpool = np.array([len(x) for x in s.gold_idx])
        print(f"        relevant per query: mean {g.mean():.1f}, min {g.min()}, max {g.max()} | in the pool: {100 * inpool.sum() / max(1, g.sum()):.1f}% "
              f"| queries with no relevant case in the pool: {(inpool == 0).sum()} | query itself in its pool: {len(s.self_pairs)}")
    else:
        print(f"        query itself in its pool: {len(s.self_pairs)}")
dv = L.Split("dev")
S = L.bm25_matrix(dv, **C["bm25"]).copy()
dv.mask_self(S)
top = np.argsort(-S, axis=1)[:, :L.K]
f1, p, r = L.micro_f1(top, dv.gold_idx, dv.n_gold)
print(f"\nBM25 only over Phase-1 tokens, dev: Micro-F1@7 {f1:.4f} (precision {p:.4f}, recall {r:.4f}) <- the lexical floor")
print("train example query id:", L.Split("train").q_ids[0], "| test example id:", L.Split("test").q_ids[0])
assert len(set(L.Split("test").q_ids)) == len(L.Split("test").q_ids), "test query ids repeat"
for arch in sorted({C["experiments"][e]["arch"] for e in SESSION_EXPERIMENTS}):
    try:
        h = L.load_holder(arch, C["checkpoints"][arch])
        e = next(x for x in SESSION_EXPERIMENTS if C["experiments"][x]["arch"] == arch)
        ex = {**C["defaults"], **C["experiments"][e]}
        print(f"{arch:8s}: window {L.window_size(h.lm, L.n_prompt_of(ex), ex['win_tokens'])} tokens, d_model {h.d_model}")
    except Exception as e:
        print(f"{arch:8s}: MISSING -> {e}")

## N9. CPU smoke test: training + embedding of the session's experiments (tiny random models, a few documents)

In [ ]:
# N9 — every experiment trains (a few steps on tiny random models) and embeds: each must end with exit code 0.
# A distillation student smoke-trains after its teacher.
import json, subprocess, sys, importlib
import pcr_common as L
importlib.reload(L)
C = json.load(open("/kaggle/working/run_config_pcr.json"))
smoke = []
for exp in SMOKE_EXPERIMENTS:
    t = C["experiments"][exp].get("teacher")
    smoke += [x for x in ([t] if t else []) + [exp] if x not in smoke]
for exp in smoke:
    for script, flag in (("train_pcr.py", "--exp"), ("embed_pcr.py", "--run")):
        r = subprocess.run([sys.executable, script, flag, exp, "--device", "cpu", "--smoke"],
                           capture_output=True, text=True, cwd="/kaggle/working")
        lines = [l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "warn(" not in l]
        print(f"===== {script} {exp}: exit code {r.returncode}")
        print("\n".join(lines[-2:] if r.returncode == 0 else lines[-25:]))

## N10. Train the session's experiments, then embed every run — on both GPUs

In [ ]:
# N10 — one job per GPU at a time. Jobs: train every SESSION_EXPERIMENT, then embed every EMBED_RUN (a run's embedding
# waits for its training; a distillation student waits for its teacher). Progress every 5 minutes.
import subprocess, sys, time, os, json, torch
C = json.load(open("/kaggle/working/run_config_pcr.json"))
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
pending = [("train", e) for e in SESSION_EXPERIMENTS] + [("embed", r) for r in EMBED_RUNS]
running, finished = {}, {}

def ready(job):
    kind, name = job
    if kind == "embed":
        waits = [("train", name)]
    else:
        e = C["experiments"][name]
        waits = [("train", x) for x in (e.get("teacher"),) if x]
    busy = set(pending) | set(running.values())
    return all(w not in busy for w in waits)

last_print = 0
while pending or running:
    for dev in devices:
        if dev in running and running[dev][2].poll() is not None:
            job = running.pop(dev)
            finished[job[:2]] = job[2].returncode
            print(time.strftime("%H:%M"), f"| {job[0]} {job[1]} finished on {dev} with exit code {job[2].returncode}", flush=True)
        if dev not in running:
            nxt = next((j for j in pending if ready(j)), None)
            if nxt:
                pending.remove(nxt)
                if nxt[0] == "embed" and finished.get(("train", nxt[1]), 0) != 0:
                    print(f"skipping embedding of {nxt[1]}: its training failed"); continue
                if nxt[0] == "train" and C["experiments"][nxt[1]].get("teacher") and \
                        finished.get(("train", C["experiments"][nxt[1]]["teacher"]), 0) != 0:
                    print(f"skipping {nxt[1]}: its teacher failed to train"); finished[nxt] = 1; continue
                out = open(f"/kaggle/working/{nxt[0]}_{nxt[1]}.out", "w")
                cmd = ["train_pcr.py", "--exp"] if nxt[0] == "train" else ["embed_pcr.py", "--run"]
                running[dev] = (*nxt, subprocess.Popen([sys.executable, cmd[0], cmd[1], nxt[1], "--device", dev],
                                                        stdout=out, stderr=subprocess.STDOUT, cwd="/kaggle/working"))
                print(time.strftime("%H:%M"), f"| started {nxt[0]} {nxt[1]} on {dev}", flush=True)
    if time.time() - last_print > 300 and running:
        for dev, (kind, name, _) in running.items():
            lines = [l for l in open(f"/kaggle/working/{kind}_{name}.out").read().splitlines() if l.strip()]
            print(time.strftime("%H:%M"), "|", kind, "|", (lines[-1] if lines else f"{name}: starting...")[:220], flush=True)
        last_print = time.time()
    time.sleep(15)
print("\nexit codes:", {f"{k} {n}": c for (k, n), c in finished.items()} or "nothing ran this session")
for (kind, name), code_ in finished.items():
    if code_ != 0 and os.path.exists(f"/kaggle/working/{kind}_{name}.out"):
        print(f"\n===== {kind} {name} FAILED — last lines:\n" + open(f"/kaggle/working/{kind}_{name}.out").read()[-3000:])

In [ ]:
# N10b — what each run reached (dev re-ranking Micro-F1@7 of the kept checkpoint, size, time, memory)
import json, os
import pcr_common as L
rows = [json.load(open(f"{f}/summary.json")) for e, f in L.find_runs("summary.json").items()]
print(f"{'run':20s} {'arch':8s} {'method':8s} {'train%':>7s} {'val F1':>7s} {'params':>8s} {'win':>4s} {'epochs':>7s} {'min':>5s} {'GPU GB':>6s}")
for s in sorted(rows, key=lambda s: -(s.get("val_f1") or 0)):
    print(f"{s['exp']:20s} {s['arch']:8s} {s['method']:8s} {100 * s['params_trainable'] / s['params_total']:7.2f} "
          f"{s['val_f1']:7.4f} {s['params_total'] / 1e6:7.1f}M {s['window']:4d} {s['epochs_done']:7.2f} "
          f"{s['train_minutes']:5.0f} {str(s['peak_gpu_gb']):>6s}")

## N11. The lab — score every variant on dev, fuse, ensemble, zips, index.csv, score chart

In [ ]:
# N11 — the lab (GPU, ~10-20 min): window aggregations x CSLS for every embedded run, the best of each run fused with
# BM25, ensembles of the k best runs, the BM25-only reference; writes the TOP_N_ZIPS best zips (+ best of each run),
# index.csv and the score chart.
import importlib, pcr_common as L, pcr_lab
importlib.reload(L); importlib.reload(pcr_lab)
lab = pcr_lab.Lab()
for run, d in lab.collect().items():
    try:
        lab.run_dense(run, d)
    except ValueError as e:
        print("skipped", run, "->", e)
lab.finish_variants()
rows = lab.finish(TOP_N_ZIPS)
pcr_lab.table(rows, 45)
pcr_lab.chart(rows, "/kaggle/working/submissions/score_chart.png")
from IPython.display import Image, display
display(Image("/kaggle/working/submissions/score_chart.png"))

## N12. Check the zips + confirm the lab's scores with the official evaluator

In [ ]:
# N12 — check every zip (format, 7 unique candidates per query, all from the test pool) and confirm the lab's dev scores
# with the official evaluator (evaluate_phase7.py) on the 3 best written variants.
import zipfile, json, glob, csv, subprocess, sys
import pcr_common as L
te = L.Split("test")
test_ids, pool = te.q_ids, set(te.c_ids)
zips = sorted(glob.glob("/kaggle/working/submissions/*.zip"))
bad = 0
for z in zips:
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    ok = (names == ["prediction.json"] and isinstance(pred, dict) and set(pred) == set(test_ids)
          and all(isinstance(v, list) and len(v) == 7 and len(set(v)) == 7 and set(v) <= pool for v in pred.values()))
    bad += not ok
    if not ok:
        print("PROBLEM", z)
print(f"{len(zips)} zips checked, {bad} with problems")
V = "/kaggle/working/submissions/dev_predictions"
rows = [r for r in csv.DictReader(open("/kaggle/working/submissions/index.csv", encoding="utf-8")) if r["file"].endswith(".zip")]
for r in rows[:3]:
    out = subprocess.run([sys.executable, "evaluate_phase7.py", "--truth", f"{V}/dev_truth.json", "--prediction",
                          f"{V}/{r['name']}.json", "--candidate-ids", f"{V}/dev_candidate_ids.json"],
                         capture_output=True, text=True)
    got = [l for l in out.stdout.splitlines() if "Micro-F1@7" in l]
    print(f"{r['name'][:70]:70s} lab dev F1 {r['dev_f1']} | official {got[0] if got else (out.stdout + out.stderr).strip()[-200:]}")

## N13. Material for the analysis: aggregation, hubness, fusion, ensembles, efficiency (tables only)

In [ ]:
# N13 — tables from index.csv (no conclusions drawn — that part is yours): mean dev F1 per run x window aggregation and
# per run x CSLS (dense variants only), dense-only vs fused with BM25 vs the BM25 reference, F1 by BM25 weight, ensembles by
# k, and size / time per run.
import csv, json, glob, numpy as np
from collections import defaultdict
rows = list(csv.DictReader(open("/kaggle/working/submissions/index.csv", encoding="utf-8")))
f = lambda r: float(r["dev_f1"])
dense = [r for r in rows if r["kind"] == "dense"]
def pivot(sel, rk, ck, title):
    t = defaultdict(lambda: defaultdict(list))
    for r in sel:
        t[r[rk]][r[ck]].append(f(r))
    cols = sorted({r[ck] for r in sel}, key=lambda x: (len(x), x))
    print(f"\n{title}")
    print(f"{rk:22s} " + " ".join(f"{c[:12]:>12s}" for c in cols))
    for run, d in sorted(t.items()):
        print(f"{run:22s} " + " ".join(f"{np.mean(d[c]):12.4f}" if c in d else f"{'':>12s}" for c in cols))
pivot(dense, "run", "agg", "mean dev F1@7 by run x window aggregation (over the CSLS strengths):")
pivot(dense, "run", "csls", "mean dev F1@7 by run x CSLS strength (over the aggregations):")
fused = [r for r in rows if r["kind"] == "dense+bm25"]
pivot(fused, "run", "bm25_w", "mean dev F1@7 of the best dense variant of each run fused with BM25, by BM25 weight:")
ref = next((r for r in rows if r["kind"] == "reference"), None)
print("\nper run: best dense-only | best fused with BM25 | BM25-only reference", f"{f(ref):.4f}" if ref else "")
runs = sorted({r["run"] for r in rows if r["kind"] in ("dense", "dense+bm25")})
for run in runs:
    d = max((f(r) for r in dense if r["run"] == run), default=0)
    u = max((f(r) for r in fused if r["run"] == run), default=0)
    x = next((r for r in dense if r["run"] == run), None)
    print(f"  {run:22s} dense {d:.4f} | fused {u:.4f} | {x['method'] if x else ''} | trainable {x['trainable_pct'] if x else ''}% | params {x['params_total'] if x else ''}")
ens = [r for r in rows if r["kind"] == "ensemble"]
print("\nensembles of the k best runs (dense only):", {r["name"]: f(r) for r in ens})
print("\ncost per run (embedding the dev + test pools):")
for p in sorted(glob.glob("/kaggle/working/pcr_emb/*/meta.json")):
    m = json.load(open(p))
    print(f"  {m['run']:22s} {m['seconds']:6.0f} s | windows {sum(v['windows'] for v in m['info'].values()):,} | trainable {m.get('params_trainable')} of {m.get('params_total')}")